In [ ]:
# =============================================================
# STEP 00 - Download e cache da base Yahoo Finance
# -------------------------------------------------------------
# Objetivo : Baixar precos ajustados de fechamento para todos
#            os tickers usados nos steps 06-14 e salvar como
#            cache CSV (formato longo) para auditoria e
#            exploracao nos steps seguintes.
# Entrada  : Yahoo Finance (download automatico via yfinance)
# Saida    : output/base_step_00_yahoo_finance.csv
# Depende  : -- (primeiro step do pre-pipeline, requer internet)
# =============================================================
import pandas as pd
import numpy as np
import yfinance as yf
import os

file_output = 'output/base_step_00_yahoo_finance.csv'
os.makedirs('output', exist_ok=True)

# Universo completo de tickers usados nos steps 06-14
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN',   # steps 06, 07 (Nash, Stackelberg)
    'JPM',  'JNJ',  'XOM',  'BRK-B',   # step 08 (Pareto / fronteira eficiente)
    'SPY',  'EEM',  'TLT',  'GLD',      # steps 13, 14 (Robusta, Estocastica)
    'EWJ',  'EWZ',  'IEF',  'VEA',      # steps 13, 14
]
START = '2015-01-01'
END   = '2023-12-31'

print(f"Baixando dados do Yahoo Finance ({START} a {END})...")
print(f"Tickers ({len(TICKERS)}): {', '.join(TICKERS)}\n")

raw    = yf.download(TICKERS, start=START, end=END, progress=False, auto_adjust=True)
precos = raw['Close'].dropna(axis=1, how='all')
tickers_validos = list(precos.columns)
print(f"Tickers validos apos download: {tickers_validos}\n")

# Converter para formato longo (tidy): Data | Ticker | Close | Retorno_Diario
# Formato longo facilita auditoria e tabelas descritivas.
frames = []
for ticker in tickers_validos:
    serie = precos[ticker].dropna().reset_index()
    serie.columns = ['Data', 'Close']
    serie['Ticker']          = ticker
    serie['Retorno_Diario']  = serie['Close'].pct_change()
    frames.append(serie[['Data', 'Ticker', 'Close', 'Retorno_Diario']])

df = pd.concat(frames, ignore_index=True)
df['Data'] = df['Data'].astype(str)

total_obs       = len(df)
total_tickers   = df['Ticker'].nunique()
periodo_inicio  = df['Data'].min()
periodo_fim     = df['Data'].max()
nulos_retorno   = df['Retorno_Diario'].isna().sum()

resultados_df = pd.DataFrame({
    'Metrica':  ['Observacoes', 'Tickers validos', 'Periodo inicio', 'Periodo fim',
                 'Nulos Retorno_Diario'],
    'Valor':    [f'{total_obs:,}', total_tickers, periodo_inicio, periodo_fim,
                 f'{nulos_retorno} (1o registro de cada serie)'],
})
print(resultados_df.to_string(index=False))

df.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo em: {file_output}")

In [ ]:
# =============================================================
# STEP 01 - Download e cache da base World Bank API
# -------------------------------------------------------------
# Objetivo : Baixar tres indicadores macroeconomicos do Banco
#            Mundial para o conjunto de paises usados nos
#            steps 09 (DEA) e 15 (Wasserstein) e salvar como
#            cache CSV para auditoria e exploracao.
# Entrada  : World Bank API (download automatico via requests)
# Saida    : output/base_step_01_world_bank.csv
# Depende  : -- (requer internet)
# =============================================================
import pandas as pd
import requests
import os

file_output = 'output/base_step_01_world_bank.csv'
os.makedirs('output', exist_ok=True)

# Uniao dos paises usados nos steps 09 e 15
# Step 09 (DEA)        : 30 paises, 2021
# Step 15 (Wasserstein): 29 paises, 2000-2022
PAISES = [
    'USA', 'CAN', 'MEX',                              # America do Norte (3)
    'BRA', 'ARG', 'CHL', 'COL', 'PER',               # America do Sul (5)
    'DEU', 'FRA', 'GBR', 'ITA', 'ESP',
    'NLD', 'SWE', 'NOR', 'POL', 'CHE', 'BEL',        # Europa (11)
    'CHN', 'JPN', 'KOR', 'IND', 'IDN',
    'PHL', 'THA', 'SGP', 'MYS',                       # Asia (9)
    'NGA', 'ZAF', 'EGY', 'ETH', 'GHA',               # Africa (5)
]
INDICADORES = {
    'NY.GDP.PCAP.CD': 'PIB_pc_USD',
    'NE.GFI.TOTL.ZS': 'Capital_pct_PIB',
    'SL.TLF.TOTL.IN': 'Forca_Trabalho',
}
ANO_INICIO = 2000
ANO_FIM    = 2022

paises_str = ';'.join(PAISES)
frames = []

for codigo, nome in INDICADORES.items():
    print(f"Baixando: {codigo} ({nome})...")
    url = (
        f'https://api.worldbank.org/v2/country/{paises_str}'
        f'/indicator/{codigo}?format=json'
        f'&date={ANO_INICIO}:{ANO_FIM}&per_page=3000'
    )
    r = requests.get(url, timeout=90)
    if r.status_code != 200:
        print(f"  ERRO: status HTTP {r.status_code}")
        continue
    dados_raw = r.json()
    if len(dados_raw) < 2:
        print("  ERRO: resposta vazia da API")
        continue
    n_entradas = 0
    for entry in dados_raw[1]:
        if entry.get('value') is not None:
            frames.append({
                'ISO':       entry['countryiso3code'],
                'Pais':      entry['country']['value'],
                'Ano':       int(entry['date']),
                'Indicador': nome,
                'Valor':     float(entry['value']),
            })
            n_entradas += 1
    print(f"  OK — {n_entradas:,} observacoes")

df_long = pd.DataFrame(frames)

# Pivotar para formato wide: ISO | Pais | Ano | PIB_pc_USD | Capital_pct_PIB | Forca_Trabalho
df = df_long.pivot_table(
    index=['ISO', 'Pais', 'Ano'],
    columns='Indicador',
    values='Valor'
).reset_index()
df.columns.name = None

total_obs    = len(df)
total_paises = df['ISO'].nunique()
total_anos   = df['Ano'].nunique()
total_nulos  = df.isna().sum().sum()

resultados_df = pd.DataFrame({
    'Metrica':  ['Observacoes (pais x ano)', 'Paises unicos', 'Anos cobertos',
                 'Ano inicio', 'Ano fim', 'Nulos total'],
    'Valor':    [f'{total_obs:,}', total_paises,
                 total_anos, df['Ano'].min(), df['Ano'].max(),
                 f'{total_nulos:,}'],
})
print(f"\n{resultados_df.to_string(index=False)}")

df.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo em: {file_output}")

In [ ]:
# =============================================================
# STEP 02 - Download e cache da base UCI Online Retail II
# -------------------------------------------------------------
# Objetivo : Baixar o dataset Online Retail II do UCI Machine
#            Learning Repository, extrair o sheet 'Year 2010-2011'
#            do arquivo Excel e salvar como cache CSV para
#            auditoria e exploracao nos steps seguintes.
# Entrada  : UCI Repository (download automatico ~45 MB ZIP)
# Saida    : output/base_step_02_online_retail.csv
# Depende  : -- (requer internet; pode demorar 1-3 min)
# =============================================================
import pandas as pd
import requests
import zipfile
import io
import os

file_output = 'output/base_step_02_online_retail.csv'
os.makedirs('output', exist_ok=True)

URL_ZIP = 'https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip'
SHEET   = 'Year 2010-2011'

print("Baixando Online Retail II do UCI Repository (~45 MB)...")
print(f"URL: {URL_ZIP}\n")

r = requests.get(URL_ZIP, timeout=300)
tamanho_mb = len(r.content) / 1_048_576
print(f"Download concluido: status {r.status_code} | {tamanho_mb:.1f} MB")

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    arquivos_zip = z.namelist()
    print(f"Arquivos no ZIP: {arquivos_zip}")
    xlsx_nome = next(f for f in arquivos_zip if f.endswith('.xlsx'))
    print(f"Lendo: {xlsx_nome} | Sheet: '{SHEET}'...")
    with z.open(xlsx_nome) as f:
        df = pd.read_excel(f, sheet_name=SHEET, engine='openpyxl')

# Normalizar nome de colunas (remover espacos extras)
df.columns = [c.strip() for c in df.columns]

# Converter InvoiceDate para string para compatibilidade CSV
df['InvoiceDate'] = df['InvoiceDate'].astype(str)

total_original   = len(df)
total_colunas    = len(df.columns)
total_nulos      = df.isna().sum().sum()
clientes_unicos  = df['Customer ID'].nunique()
produtos_unicos  = df['StockCode'].nunique()
paises_unicos    = df['Country'].nunique()
data_inicio      = df['InvoiceDate'].min()
data_fim         = df['InvoiceDate'].max()

resultados_df = pd.DataFrame({
    'Metrica':  ['Registros brutos', 'Colunas', 'Nulos total',
                 'Clientes unicos', 'Produtos unicos', 'Paises',
                 'Data inicio', 'Data fim'],
    'Valor':    [f'{total_original:,}', total_colunas, f'{total_nulos:,}',
                 f'{clientes_unicos:,}', f'{produtos_unicos:,}', paises_unicos,
                 data_inicio, data_fim],
})
print(f"\n{resultados_df.to_string(index=False)}")

df.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo em: {file_output}")

In [ ]:
# =============================================================
# STEP 03 - Auditoria das tres bases de dados
# -------------------------------------------------------------
# Objetivo : Auditar as tres bases do pipeline (Yahoo Finance,
#            World Bank API e UCI Online Retail II), verificando
#            nulos por coluna, duplicatas, tipos de dado e taxa
#            de aproveitamento apos dropna(). Produz um relatorio
#            consolidado unico cobrindo as tres fontes.
# Entrada  : output/base_step_00_yahoo_finance.csv
#            output/base_step_01_world_bank.csv
#            output/base_step_02_online_retail.csv
# Saida    : output/base_step_03_auditoria.csv
# Depende  : STEPS 00, 01, 02
# =============================================================
import pandas as pd
import numpy as np
import os

file_output = 'output/base_step_03_auditoria.csv'
os.makedirs('output', exist_ok=True)

BASES = {
    'Yahoo Finance':     'output/base_step_00_yahoo_finance.csv',
    'World Bank API':    'output/base_step_01_world_bank.csv',
    'UCI Online Retail': 'output/base_step_02_online_retail.csv',
}


def print_separador(titulo):
    linha = '=' * 65
    print(f"\n{linha}")
    print(f"  {titulo}")
    print(linha)


def print_tabela_bonita(df):
    cols = list(df.columns)
    df_print = df.copy()
    for c in cols:
        df_print[c] = df_print[c].astype(str).str.replace('\n', ' ')
    col_widths = {c: max(df_print[c].str.len().max(), len(c)) for c in cols}
    header = ' ' + '  '.join(c.ljust(col_widths[c]) for c in cols) + ' '
    print(header)
    for _, row in df_print.iterrows():
        linha = ' ' + '  '.join(str(row[c]).ljust(col_widths[c]) for c in cols) + ' '
        print(linha)


# -------------------------------------------------------
resumo_geral  = []
linhas_colunas = []

for nome_base, caminho in BASES.items():
    print_separador(f'BASE: {nome_base}')
    print(f"  Arquivo: {caminho}")

    df = pd.read_csv(caminho, sep=';', decimal=',',
                     encoding='utf-8-sig', low_memory=False)

    n_linhas, n_colunas = df.shape
    n_duplicatas        = df.duplicated().sum()
    n_nulos_total       = df.isna().sum().sum()
    pct_nulos_celulas   = (n_nulos_total / (n_linhas * n_colunas)) * 100
    df_sem_nulos        = df.dropna()
    n_aproveitados      = len(df_sem_nulos)
    pct_aproveitamento  = (n_aproveitados / n_linhas) * 100

    print(f"\n  Forma (linhas x colunas) : {n_linhas:,} x {n_colunas}")
    print(f"  Duplicatas               : {n_duplicatas:,}")
    print(f"  Celulas nulas            : {n_nulos_total:,}  "
          f"({pct_nulos_celulas:.1f}% das {n_linhas * n_colunas:,} celulas)")
    print(f"  Registros apos dropna()  : {n_aproveitados:,}  "
          f"({pct_aproveitamento:.1f}% do total)")

    # Detalhe por coluna
    print(f"\n  Auditoria por coluna:")
    tbl_cols = []
    for col in df.columns:
        n_nulos  = int(df[col].isna().sum())
        pct_col  = round((n_nulos / n_linhas) * 100, 1)
        n_unicos = df[col].nunique()
        dtype    = str(df[col].dtype)
        exemplo  = str(df[col].dropna().iloc[0]) if df[col].dropna().shape[0] > 0 else 'N/A'
        if len(exemplo) > 30:
            exemplo = exemplo[:27] + '...'
        row_col = {
            'Base':      nome_base,
            'Coluna':    col,
            'Dtype':     dtype,
            'Nulos':     n_nulos,
            'Nulos_%':   pct_col,
            'Unicos':    n_unicos,
            'Exemplo':   exemplo,
        }
        tbl_cols.append(row_col)
        linhas_colunas.append(row_col)

    print_tabela_bonita(
        pd.DataFrame(tbl_cols)[['Coluna', 'Dtype', 'Nulos', 'Nulos_%', 'Unicos', 'Exemplo']]
    )

    resumo_geral.append({
        'Base':               nome_base,
        'Linhas':             n_linhas,
        'Colunas':            n_colunas,
        'Duplicatas':         n_duplicatas,
        'Nulos_total':        n_nulos_total,
        'Nulos_pct_celulas':  round(pct_nulos_celulas, 1),
        'Aproveitados':       n_aproveitados,
        'Aproveitamento_pct': round(pct_aproveitamento, 1),
    })


# Resumo consolidado das tres bases
print_separador('RESUMO CONSOLIDADO DAS TRES BASES')
df_resumo = pd.DataFrame(resumo_geral)
print_tabela_bonita(df_resumo)

# Salvar detalhes por coluna (todas as bases em um unico CSV)
df_saida = pd.DataFrame(linhas_colunas)
df_saida.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo em: {file_output}")

In [ ]:
# =============================================================
# STEP 04 - Tabelas de variaveis qualitativas e quantitativas
# -------------------------------------------------------------
# Objetivo : Identificar e descrever as variaveis qualitativas
#            (moda, numero de categorias) e quantitativas
#            (min, max, media, quartis, mediana) das tres bases
#            do pipeline em um unico step consolidado.
# Entrada  : output/base_step_00_yahoo_finance.csv
#            output/base_step_01_world_bank.csv
#            output/base_step_02_online_retail.csv
# Saida    : output/base_step_04_qualitativas.csv
#            output/base_step_04_quantitativas.csv
# Depende  : STEPS 00, 01, 02
# =============================================================
import pandas as pd
import numpy as np
import os

file_quali  = 'output/base_step_04_qualitativas.csv'
file_quanti = 'output/base_step_04_quantitativas.csv'
os.makedirs('output', exist_ok=True)

BASES = {
    'Yahoo Finance':     'output/base_step_00_yahoo_finance.csv',
    'World Bank API':    'output/base_step_01_world_bank.csv',
    'UCI Online Retail': 'output/base_step_02_online_retail.csv',
}

MAX_CATS_EXIBIDAS = 8   # maximo de categorias distintas exibidas na tabela


def como_numero(serie):
    """Converte serie para numerico, tratando virgula decimal."""
    return pd.to_numeric(
        serie.dropna().astype(str).str.replace(',', '.', regex=False),
        errors='coerce'
    )


def eh_qualitativa(serie):
    """Retorna True se NENHUM valor da serie converte para numero."""
    num = como_numero(serie)
    return num.notna().sum() == 0


def eh_quantitativa(serie):
    """Retorna True se TODOS os valores nao-nulos da serie sao numericos."""
    if serie.dropna().empty:
        return False
    return como_numero(serie).notna().sum() == len(serie.dropna())


def print_separador(titulo):
    linha = '=' * 65
    print(f"\n{linha}")
    print(f"  {titulo}")
    print(linha)


def print_tabela_bonita(df):
    cols = list(df.columns)
    df_print = df.copy()
    for c in cols:
        df_print[c] = df_print[c].astype(str).str.replace('\n', ' ')
    col_widths = {c: max(df_print[c].str.len().max(), len(c)) for c in cols}
    header = ' ' + '  '.join(c.ljust(col_widths[c]) for c in cols) + ' '
    print(header)
    for _, row in df_print.iterrows():
        linha = ' ' + '  '.join(str(row[c]).ljust(col_widths[c]) for c in cols) + ' '
        print(linha)


# -------------------------------------------------------
todas_quali  = []
todas_quanti = []

for nome_base, caminho in BASES.items():
    df = pd.read_csv(caminho, sep=';', decimal=',',
                     encoding='utf-8-sig', low_memory=False)

    qualitativas  = [c for c in df.columns if eh_qualitativa(df[c])]
    quantitativas = [c for c in df.columns if eh_quantitativa(df[c])]

    # --------------------------------------------------
    # TABELA DE VARIAVEIS QUALITATIVAS
    # --------------------------------------------------
    print_separador(f'{nome_base}  —  VARIAVEIS QUALITATIVAS ({len(qualitativas)})')

    if qualitativas:
        linhas_q = []
        for col in qualitativas:
            serie        = df[col].dropna()
            n_cats       = serie.nunique()
            moda         = serie.mode().iloc[0] if not serie.mode().empty else 'N/A'
            freq_moda    = int((serie == moda).sum())
            pct_moda     = round(freq_moda / len(serie) * 100, 1)
            cats_lista   = sorted(serie.unique().astype(str))
            cats_exib    = ', '.join(cats_lista[:MAX_CATS_EXIBIDAS])
            if n_cats > MAX_CATS_EXIBIDAS:
                cats_exib += f'  ... (+{n_cats - MAX_CATS_EXIBIDAS})'
            linhas_q.append({
                'Base':        nome_base,
                'Variavel':    col,
                'N_Categorias': n_cats,
                'Moda':        str(moda),
                'Freq_Moda_%': pct_moda,
                'Categorias':  cats_exib,
            })
            todas_quali.append(linhas_q[-1])

        print_tabela_bonita(
            pd.DataFrame(linhas_q)[['Variavel', 'N_Categorias', 'Moda',
                                     'Freq_Moda_%', 'Categorias']]
        )
    else:
        print("  (nenhuma variavel qualitativa detectada nesta base)")

    # --------------------------------------------------
    # TABELA DE VARIAVEIS QUANTITATIVAS
    # --------------------------------------------------
    print_separador(f'{nome_base}  —  VARIAVEIS QUANTITATIVAS ({len(quantitativas)})')

    if quantitativas:
        linhas_n = []
        for col in quantitativas:
            s = como_numero(df[col])
            linhas_n.append({
                'Base':       nome_base,
                'Variavel':   col,
                'N_obs':      int(s.count()),
                'Min.':       round(s.min(), 4),
                'Max.':       round(s.max(), 4),
                'Media':      round(s.mean(), 4),
                '1o_Quartil': round(s.quantile(0.25), 4),
                'Mediana':    round(s.median(), 4),
                '3o_Quartil': round(s.quantile(0.75), 4),
                'Desvio_Pad': round(s.std(), 4),
            })
            todas_quanti.append(linhas_n[-1])

        pd.options.display.float_format = lambda x: f'{x:,.4f}'
        print_tabela_bonita(
            pd.DataFrame(linhas_n)[['Variavel', 'N_obs', 'Min.', 'Max.',
                                     'Media', '1o_Quartil', 'Mediana',
                                     '3o_Quartil', 'Desvio_Pad']]
        )
    else:
        print("  (nenhuma variavel quantitativa detectada nesta base)")


# -------------------------------------------------------
# Salvar CSVs consolidados
# -------------------------------------------------------
pd.DataFrame(todas_quali).to_csv(
    file_quali, sep=';', index=False, encoding='utf-8-sig', decimal=','
)
pd.DataFrame(todas_quanti).to_csv(
    file_quanti, sep=';', index=False, encoding='utf-8-sig', decimal=','
)

print(f"\nArquivos salvos em:")
print(f"  {file_quali}")
print(f"  {file_quanti}")

In [ ]:
# =============================================================
# STEP 05 - Dicionario de variaveis e pre-processamento
# -------------------------------------------------------------
# Objetivo : Gerar um dicionario de variaveis consolidado para
#            as tres bases (tipo, nulos%, unicos, exemplo) e
#            produzir versoes limpas (sem nulos criticos, sem
#            duplicatas) prontas para consumo pelos steps 06-16.
# Entrada  : output/base_step_00_yahoo_finance.csv
#            output/base_step_01_world_bank.csv
#            output/base_step_02_online_retail.csv
# Saida    : output/base_step_05_dicionario.csv
#            output/base_step_05_yahoo_limpa.csv
#            output/base_step_05_world_bank_limpa.csv
#            output/base_step_05_online_retail_limpa.csv
# Depende  : STEPS 00, 01, 02
# =============================================================
import pandas as pd
import numpy as np
import os

file_dicionario  = 'output/base_step_05_dicionario.csv'
file_yf_limpa    = 'output/base_step_05_yahoo_limpa.csv'
file_wb_limpa    = 'output/base_step_05_world_bank_limpa.csv'
file_ori_limpa   = 'output/base_step_05_online_retail_limpa.csv'
os.makedirs('output', exist_ok=True)


def como_numero(serie):
    return pd.to_numeric(
        serie.dropna().astype(str).str.replace(',', '.', regex=False),
        errors='coerce'
    )


def classificar_tipo(serie_bruta, serie_limpa):
    """Classifica a variavel como Qualitativa ou Quantitativa."""
    num = como_numero(serie_limpa)
    return 'Quantitativa' if num.notna().all() else 'Qualitativa'


def print_tabela_bonita(df):
    cols = list(df.columns)
    df_print = df.copy()
    for c in cols:
        df_print[c] = df_print[c].astype(str).str.replace('\n', ' ')
    col_widths = {c: max(df_print[c].str.len().max(), len(c)) for c in cols}
    header = ' ' + '  '.join(c.ljust(col_widths[c]) for c in cols) + ' '
    print(header)
    for _, row in df_print.iterrows():
        linha = ' ' + '  '.join(str(row[c]).ljust(col_widths[c]) for c in cols) + ' '
        print(linha)


# -------------------------------------------------------
dicionario = []


# --- Yahoo Finance ---
print("=" * 65)
print("  Pre-processamento: Yahoo Finance")
print("=" * 65)
df_yf = pd.read_csv('output/base_step_00_yahoo_finance.csv',
                    sep=';', decimal=',', encoding='utf-8-sig')

# Limpar: remover nulos (primeiro retorno de cada serie) e duplicatas
df_yf_limpa = df_yf.dropna().drop_duplicates()

n_bruto = len(df_yf)
n_limpo = len(df_yf_limpa)
print(f"  Bruto   : {n_bruto:,} registros")
print(f"  Limpo   : {n_limpo:,} registros  "
      f"(removidos: {n_bruto - n_limpo:,} — {(n_bruto - n_limpo) / n_bruto * 100:.1f}%)")

for col in df_yf.columns:
    s_bruta = df_yf[col]
    s_limpa = df_yf_limpa[col] if col in df_yf_limpa.columns else s_bruta
    exemplo = str(s_bruta.dropna().iloc[0]) if s_bruta.dropna().shape[0] > 0 else 'N/A'
    if len(exemplo) > 25:
        exemplo = exemplo[:22] + '...'
    dicionario.append({
        'Base':        'Yahoo Finance',
        'Variavel':    col,
        'Tipo':        classificar_tipo(s_bruta, s_limpa),
        'Dtype_pandas': str(s_bruta.dtype),
        'Nulos_pct':   round(s_bruta.isna().mean() * 100, 1),
        'Unicos':      s_bruta.nunique(),
        'Exemplo':     exemplo,
    })


# --- World Bank API ---
print("\n" + "=" * 65)
print("  Pre-processamento: World Bank API")
print("=" * 65)
df_wb = pd.read_csv('output/base_step_01_world_bank.csv',
                    sep=';', decimal=',', encoding='utf-8-sig')

# Limpar: manter apenas registros com os tres indicadores completos
df_wb_limpa = df_wb.dropna().drop_duplicates()

n_bruto = len(df_wb)
n_limpo = len(df_wb_limpa)
print(f"  Bruto   : {n_bruto:,} registros (pais x ano)")
print(f"  Limpo   : {n_limpo:,} registros  "
      f"(removidos: {n_bruto - n_limpo:,} — {(n_bruto - n_limpo) / n_bruto * 100:.1f}%)")

for col in df_wb.columns:
    s_bruta = df_wb[col]
    s_limpa = df_wb_limpa[col] if col in df_wb_limpa.columns else s_bruta
    exemplo = str(s_bruta.dropna().iloc[0]) if s_bruta.dropna().shape[0] > 0 else 'N/A'
    if len(exemplo) > 25:
        exemplo = exemplo[:22] + '...'
    dicionario.append({
        'Base':        'World Bank API',
        'Variavel':    col,
        'Tipo':        classificar_tipo(s_bruta, s_limpa),
        'Dtype_pandas': str(s_bruta.dtype),
        'Nulos_pct':   round(s_bruta.isna().mean() * 100, 1),
        'Unicos':      s_bruta.nunique(),
        'Exemplo':     exemplo,
    })


# --- UCI Online Retail II ---
print("\n" + "=" * 65)
print("  Pre-processamento: UCI Online Retail II")
print("=" * 65)
df_ori = pd.read_csv('output/base_step_02_online_retail.csv',
                     sep=';', decimal=',', encoding='utf-8-sig',
                     low_memory=False)

# Limpar: remover Customer ID nulo, Quantity <= 0, Price <= 0, duplicatas
# Converter Quantity e Price para numerico antes de filtrar
df_ori['Quantity'] = pd.to_numeric(
    df_ori['Quantity'].astype(str).str.replace(',', '.'), errors='coerce'
)
df_ori['Price'] = pd.to_numeric(
    df_ori['Price'].astype(str).str.replace(',', '.'), errors='coerce'
)
df_ori_limpa = (
    df_ori[df_ori['Customer ID'].notna()]
    .loc[lambda d: d['Quantity'] > 0]
    .loc[lambda d: d['Price'] > 0]
    .drop_duplicates()
    .reset_index(drop=True)
)

n_bruto = len(df_ori)
n_limpo = len(df_ori_limpa)
print(f"  Bruto   : {n_bruto:,} registros")
print(f"  Limpo   : {n_limpo:,} registros  "
      f"(removidos: {n_bruto - n_limpo:,} — {(n_bruto - n_limpo) / n_bruto * 100:.1f}%)")
print(f"  Criterios de limpeza: Customer ID nao-nulo | Quantity > 0 | Price > 0")

for col in df_ori.columns:
    s_bruta = df_ori[col]
    s_limpa = df_ori_limpa[col] if col in df_ori_limpa.columns else s_bruta
    exemplo = str(s_bruta.dropna().iloc[0]) if s_bruta.dropna().shape[0] > 0 else 'N/A'
    if len(exemplo) > 25:
        exemplo = exemplo[:22] + '...'
    dicionario.append({
        'Base':        'UCI Online Retail',
        'Variavel':    col,
        'Tipo':        classificar_tipo(s_bruta, s_limpa),
        'Dtype_pandas': str(s_bruta.dtype),
        'Nulos_pct':   round(s_bruta.isna().mean() * 100, 1),
        'Unicos':      s_bruta.nunique(),
        'Exemplo':     exemplo,
    })


# -------------------------------------------------------
# Imprimir dicionario consolidado
# -------------------------------------------------------
print("\n\n" + "=" * 65)
print("  DICIONARIO DE VARIAVEIS CONSOLIDADO")
print("=" * 65)
df_dic = pd.DataFrame(dicionario)
print_tabela_bonita(df_dic)


# -------------------------------------------------------
# Salvar todos os arquivos
# -------------------------------------------------------
df_dic.to_csv(
    file_dicionario, sep=';', index=False, encoding='utf-8-sig', decimal=','
)
df_yf_limpa.to_csv(
    file_yf_limpa, sep=';', index=False, encoding='utf-8-sig', decimal=','
)
df_wb_limpa.to_csv(
    file_wb_limpa, sep=';', index=False, encoding='utf-8-sig', decimal=','
)
df_ori_limpa.to_csv(
    file_ori_limpa, sep=';', index=False, encoding='utf-8-sig', decimal=','
)

print(f"\nArquivos salvos em:")
print(f"  {file_dicionario}")
print(f"  {file_yf_limpa}")
print(f"  {file_wb_limpa}")
print(f"  {file_ori_limpa}")

In [ ]:
# =============================================================
# STEP 06 - Teoria dos Jogos e Equilíbrio de Nash
# -------------------------------------------------------------
# Objetivo: Modelar interações estratégicas entre empresas
#           concorrentes via Equilíbrio de Nash. Constrói a
#           matriz de payoff a partir de retornos históricos
#           e identifica estratégias de equilíbrio (puras e
#           mistas) em um jogo de competição de mercado.
# Entrada  : output/base_step_00_yahoo_finance.csv  (gerado pelo step-00)
# Saídas   : output/base_step_06_nash_equilibrio.csv
#            output/base_step_06_nash_payoff_heatmap.png
#            output/base_step_06_nash_estrategias.png
#            output/base_step_06_nash_retornos_cumulados.png
#            output/base_step06_nash_payoff_interativo.html
#            output/base_step06_nash_retornos_interativo.html
# Depende  : STEP 00
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

file_output          = 'output/base_step_06_nash_equilibrio.csv'
file_payoff_heatmap  = 'output/base_step_06_nash_payoff_heatmap.png'
file_estrategias     = 'output/base_step_06_nash_estrategias.png'
file_retornos_cumul  = 'output/base_step_06_nash_retornos_cumulados.png'
file_payoff_inter    = 'output/base_step06_nash_payoff_interativo.html'
file_retornos_inter  = 'output/base_step06_nash_retornos_interativo.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Aquisição de dados (base auditada step-00) ----------
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
print("Carregando base auditada do step-00 (Yahoo Finance)...")
df_cache = pd.read_csv('output/base_step_00_yahoo_finance.csv',
                       sep=';', decimal=',', encoding='utf-8-sig')
df_cache['Data'] = pd.to_datetime(df_cache['Data'])
mask = (df_cache['Ticker'].isin(tickers) &
        (df_cache['Data'] >= '2021-01-01') &
        (df_cache['Data'] <= '2023-12-31'))
dados = df_cache[mask].pivot(index='Data', columns='Ticker', values='Close')[tickers]
print(f"Dados carregados: {dados.shape[0]} observações, {len(tickers)} ativos")

# ---------- 2. Construção da Matriz de Payoff e Equilíbrio de Nash ----------
retornos      = dados.pct_change().dropna()
ret_anual     = retornos.mean() * 252
vol_anual     = retornos.std() * np.sqrt(252)
sharpe        = ret_anual / vol_anual
ret_acum      = (1 + retornos).cumprod()

n = len(tickers)

# Payoff do jogador A (linha) ao adotar estratégia i contra estratégia j do rival:
# payoff_A[i,j] = retorno_i - correlação(i,j) * retorno_j
# (quanto menor a correlação, menos o rival captura do ganho de i)
payoff_A = np.zeros((n, n))
payoff_B = np.zeros((n, n))
for i, t1 in enumerate(tickers):
    for j, t2 in enumerate(tickers):
        corr = retornos[t1].corr(retornos[t2])
        payoff_A[i, j] = ret_anual[t1] - corr * ret_anual[t2]
        payoff_B[i, j] = ret_anual[t2] - corr * ret_anual[t1]

# Equilíbrios de Nash em estratégias puras: (i*, j*) é Nash se
#   payoff_A[i*, j*] >= payoff_A[i, j*]  para todo i  (melhor resposta de A)
#   payoff_B[i*, j*] >= payoff_B[i*, j]  para todo j  (melhor resposta de B)
nash_equilibria = []
for i in range(n):
    for j in range(n):
        mr_A = (payoff_A[i, j] == payoff_A[:, j].max())
        mr_B = (payoff_B[i, j] == payoff_B[i, :].max())
        if mr_A and mr_B:
            nash_equilibria.append({
                'Empresa_A': tickers[i],
                'Empresa_B': tickers[j],
                'Payoff_A':  round(payoff_A[i, j], 6),
                'Payoff_B':  round(payoff_B[i, j], 6),
            })

print(f"\nEquilíbrios de Nash (estratégias puras) encontrados: {len(nash_equilibria)}")
for ne in nash_equilibria:
    print(f"  ({ne['Empresa_A']}, {ne['Empresa_B']}) → "
          f"payoff_A={ne['Payoff_A']:.4f}, payoff_B={ne['Payoff_B']:.4f}")

# Estatísticas dos ativos
df_stats = pd.DataFrame({
    'Ticker':          tickers,
    'Retorno_Anual':   [round(ret_anual[t], 6) for t in tickers],
    'Volatilidade':    [round(vol_anual[t], 6) for t in tickers],
    'Sharpe':          [round(sharpe[t], 6) for t in tickers],
    'Nash_Equilibrio': ['Sim' if any(ne['Empresa_A'] == t for ne in nash_equilibria) else 'Não'
                        for t in tickers],
})

df_nash = pd.DataFrame(nash_equilibria)
df_resultado = pd.concat([df_stats, df_nash.rename(columns={
    'Empresa_A': 'Nash_Empresa_A', 'Empresa_B': 'Nash_Empresa_B',
    'Payoff_A': 'Nash_Payoff_A',  'Payoff_B': 'Nash_Payoff_B'
})], axis=1)

# ---------- 3. Salvar resultados ----------
df_resultado.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Heatmap da Matriz de Payoff (Jogador A)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(payoff_A, annot=True, fmt='.3f', xticklabels=tickers, yticklabels=tickers,
            cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Matriz de Payoff — Jogador A')
axes[0].set_xlabel('Estratégia do Rival (Coluna)')
axes[0].set_ylabel('Estratégia Própria (Linha)')

sns.heatmap(payoff_B, annot=True, fmt='.3f', xticklabels=tickers, yticklabels=tickers,
            cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Matriz de Payoff — Jogador B')
axes[1].set_xlabel('Estratégia Própria (Coluna)')
axes[1].set_ylabel('Estratégia do Rival (Linha)')

plt.suptitle('Matrizes de Payoff — Equilíbrio de Nash', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(file_payoff_heatmap, dpi=150)
plt.show()
plt.close()

# 4.2 Espaço de Estratégias: Retorno vs Volatilidade
cores = ['#1565C0', '#C62828', '#2E7D32', '#E65100']
plt.figure(figsize=(9, 6))
for i, t in enumerate(tickers):
    plt.scatter(vol_anual[t] * 100, ret_anual[t] * 100,
                s=250, color=cores[i], label=t, zorder=5)
    plt.annotate(t, (vol_anual[t] * 100, ret_anual[t] * 100),
                 textcoords='offset points', xytext=(9, 4), fontsize=11, fontweight='bold')
for ne in nash_equilibria:
    idx = tickers.index(ne['Empresa_A'])
    plt.scatter(vol_anual[tickers[idx]] * 100, ret_anual[tickers[idx]] * 100,
                s=600, edgecolors='gold', facecolors='none', linewidths=2.5, zorder=6)
plt.xlabel('Volatilidade Anual (%)', fontsize=11)
plt.ylabel('Retorno Anual (%)', fontsize=11)
plt.title('Espaço de Estratégias: Retorno × Volatilidade\n(Nash em equilíbrio destacado em dourado)',
          fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_estrategias, dpi=150)
plt.show()
plt.close()

# 4.3 Retornos Acumulados
plt.figure(figsize=(11, 5))
for i, t in enumerate(tickers):
    plt.plot(ret_acum.index, ret_acum[t], label=t, linewidth=2, color=cores[i])
plt.title('Retorno Acumulado dos Concorrentes (2021–2023)', fontsize=12)
plt.xlabel('Data')
plt.ylabel('Retorno Acumulado (base 1)')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_retornos_cumul, dpi=150)
plt.show()
plt.close()

# 4.4 Heatmap interativo da Matriz de Payoff (Plotly)
texto = [[f'{payoff_A[i,j]:.4f}' for j in range(n)] for i in range(n)]
fig_heat = go.Figure(data=go.Heatmap(
    z=payoff_A, x=tickers, y=tickers,
    colorscale='RdBu', zmid=0,
    text=texto, texttemplate='%{text}', showscale=True
))
fig_heat.update_layout(
    title='Matriz de Payoff (Jogador A) — Interativo',
    xaxis_title='Estratégia do Rival', yaxis_title='Estratégia Própria'
)
fig_heat.write_html(file_payoff_inter)
fig_heat.show()
print(f"Heatmap interativo salvo: {file_payoff_inter}")

# 4.5 Retornos acumulados interativo (Plotly)
df_ret_plot = ret_acum.reset_index().melt(id_vars='Data', var_name='Empresa', value_name='Retorno_Acumulado')
fig_ret = px.line(df_ret_plot, x='Data', y='Retorno_Acumulado', color='Empresa',
                  title='Retorno Acumulado dos Concorrentes (Equilíbrio de Nash)',
                  labels={'Data': 'Data', 'Retorno_Acumulado': 'Retorno Acumulado'})
fig_ret.write_html(file_retornos_inter)
fig_ret.show()
print(f"Retornos interativos salvos: {file_retornos_inter}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_payoff_heatmap}")
print(f"  {file_estrategias}")
print(f"  {file_retornos_cumul}")
print(f"  {file_payoff_inter}")
print(f"  {file_retornos_inter}")


In [ ]:
# =============================================================
# STEP 07 - Equilíbrio de Stackelberg
# -------------------------------------------------------------
# Objetivo: Modelar competição líder-seguidor (Stackelberg) no
#           mercado de tecnologia. O líder anuncia quantidade
#           primeiro; os seguidores respondem de forma ótima.
#           Compara o equilíbrio de Stackelberg com o de Nash-
#           Cournot para mensurar a vantagem do primeiro mover.
# Entrada  : output/base_step_00_yahoo_finance.csv  (gerado pelo step-00)
# Saídas   : output/base_step_07_stackelberg_resultado.csv
#            output/base_step_07_stackelberg_equilibrio.png
#            output/base_step_07_stackelberg_comparativo.png
#            output/base_step_07_stackelberg_lucros.png
#            output/base_step07_stackelberg_interativo.html
#            output/base_step07_stackelberg_3d.html
# Depende  : STEP 00
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy.optimize import minimize_scalar

file_output          = 'output/base_step_07_stackelberg_resultado.csv'
file_equilibrio      = 'output/base_step_07_stackelberg_equilibrio.png'
file_comparativo     = 'output/base_step_07_stackelberg_comparativo.png'
file_lucros          = 'output/base_step_07_stackelberg_lucros.png'
file_interativo      = 'output/base_step07_stackelberg_interativo.html'
file_3d              = 'output/base_step07_stackelberg_3d.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Aquisição de dados (base auditada step-00) ----------
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
lider   = 'AAPL'
seguidores = ['MSFT', 'GOOGL', 'AMZN']
print("Carregando base auditada do step-00 (Yahoo Finance)...")
df_cache = pd.read_csv('output/base_step_00_yahoo_finance.csv',
                       sep=';', decimal=',', encoding='utf-8-sig')
df_cache['Data'] = pd.to_datetime(df_cache['Data'])
mask = (df_cache['Ticker'].isin(tickers) &
        (df_cache['Data'] >= '2021-01-01') &
        (df_cache['Data'] <= '2023-12-31'))
dados = df_cache[mask].pivot(index='Data', columns='Ticker', values='Close')[tickers]
print(f"Dados carregados: {dados.shape[0]} observações, {len(tickers)} ativos")

retornos  = dados.pct_change().dropna()
ret_anual = retornos.mean() * 252
vol_anual = retornos.std() * np.sqrt(252)

# ---------- 2. Modelo de Stackelberg ----------
# Premissa Cournot linear: demanda inversa P = a - b*(Q_total)
# Custo marginal c_i estimado como (1 - Sharpe_normalizado) * a
# Lucro_i = (P - c_i) * q_i

a = 100.0   # preço máximo (normalizado)
b = 1.0     # inclinação da demanda

sharpe     = ret_anual / vol_anual
sharpe_min = sharpe.min()
sharpe_max = sharpe.max()
# Custo marginal: empresas mais eficientes (maior Sharpe) têm custo menor
custo = {t: a * 0.2 + 0.6 * a * (1 - (sharpe[t] - sharpe_min) / (sharpe_max - sharpe_min + 1e-9))
         for t in tickers}

# --- Nash-Cournot (simultâneo) ---
# q_i* = (a - c_i - b * soma(q_j, j!=i)) / (2*b)  → iteração de ponto fixo
n = len(tickers)
q_cournot = np.ones(n) * (a / (b * (n + 1)))
for _ in range(1000):
    q_novo = np.array([
        max(0, (a - list(custo.values())[i] - b * (q_cournot.sum() - q_cournot[i])) / (2 * b))
        for i in range(n)
    ])
    if np.max(np.abs(q_novo - q_cournot)) < 1e-9:
        break
    q_cournot = q_novo.copy()

P_cournot = a - b * q_cournot.sum()
lucro_cournot = {tickers[i]: (P_cournot - list(custo.values())[i]) * q_cournot[i] for i in range(n)}

# --- Stackelberg (sequencial) ---
# Melhor resposta dos seguidores para quantidade do líder q_L:
# q_seg_i*(q_L) = (a - c_i - b*q_L - b*soma(q_outros_segs)) / (2*b)
# Em equilíbrio simétrico dos seguidores:
# q_seg_i* = (a - c_i - b*q_L) / (b*(n_segs+1))   [simplificado para segs homogêneos]
n_segs = len(seguidores)

def lucro_lider(q_L):
    soma_segs = sum(
        max(0, (a - custo[s] - b * q_L) / (b * (n_segs + 1)))
        for s in seguidores
    )
    P = a - b * (q_L + soma_segs)
    return -(P - custo[lider]) * q_L   # negativo para minimizar

res     = minimize_scalar(lucro_lider, bounds=(0, a / b), method='bounded')
q_L_st  = res.x

q_segs_st = {s: max(0, (a - custo[s] - b * q_L_st) / (b * (n_segs + 1)))
             for s in seguidores}
Q_st    = q_L_st + sum(q_segs_st.values())
P_st    = a - b * Q_st

lucro_st = {lider: (P_st - custo[lider]) * q_L_st}
lucro_st.update({s: (P_st - custo[s]) * q_segs_st[s] for s in seguidores})

print(f"\nCournot  — Preço: {P_cournot:.2f}  |  Q total: {q_cournot.sum():.2f}")
print(f"Stackelberg — Preço: {P_st:.2f}  |  Q total: {Q_st:.2f}")
print(f"\nLucro Líder ({lider})  Cournot: {lucro_cournot[lider]:.2f}  |  Stackelberg: {lucro_st[lider]:.2f}")

# ---------- 3. Salvar resultados ----------
registros = []
for t in tickers:
    idx = tickers.index(t)
    registros.append({
        'Empresa':           t,
        'Papel':             'Líder' if t == lider else 'Seguidor',
        'Custo_Marginal':    round(custo[t], 4),
        'Q_Cournot':         round(q_cournot[idx], 4),
        'Lucro_Cournot':     round(lucro_cournot[t], 4),
        'Q_Stackelberg':     round(q_L_st if t == lider else q_segs_st.get(t, 0), 4),
        'Lucro_Stackelberg': round(lucro_st[t], 4),
        'Vantagem_Lider_%':  round((lucro_st[t] / (lucro_cournot[t] + 1e-9) - 1) * 100, 2),
    })

df_resultado = pd.DataFrame(registros)
df_resultado.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Comparativo de quantidades: Cournot vs Stackelberg
x   = np.arange(len(tickers))
w   = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, [round(q_cournot[i], 2) for i in range(n)],   width=w, label='Cournot (Nash)', color='#1565C0')
ax.bar(x + w/2, [df_resultado.loc[df_resultado.Empresa == t, 'Q_Stackelberg'].values[0]
                 for t in tickers], width=w, label='Stackelberg', color='#C62828')
ax.set_xticks(x)
ax.set_xticklabels(tickers, fontsize=11)
ax.set_ylabel('Quantidade de Equilíbrio')
ax.set_title('Quantidades de Equilíbrio: Cournot vs Stackelberg', fontsize=12)
ax.legend(fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(file_equilibrio, dpi=150)
plt.show()
plt.close()

# 4.2 Lucros comparativos
lucros_cournot_list    = [lucro_cournot[t] for t in tickers]
lucros_stackelberg_list = [lucro_st[t] for t in tickers]
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, lucros_cournot_list,    width=w, label='Cournot', color='#1565C0')
ax.bar(x + w/2, lucros_stackelberg_list, width=w, label='Stackelberg', color='#C62828')
ax.set_xticks(x)
ax.set_xticklabels(tickers, fontsize=11)
ax.set_ylabel('Lucro')
ax.set_title('Lucros de Equilíbrio: Cournot vs Stackelberg\n(Líder destacado)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.4)
# Destacar líder
ax.axvspan(-0.5, 0.5, alpha=0.08, color='gold')
ax.annotate('Líder', xy=(0, max(lucros_stackelberg_list) * 0.95),
            ha='center', fontsize=9, color='#8B6914')
plt.tight_layout()
plt.savefig(file_comparativo, dpi=150)
plt.show()
plt.close()

# 4.3 Vantagem do primeiro mover (%)
# Ordenar por valor (SKILL: ordenar barras se não há ordem natural entre categorias)
df_vantagem = df_resultado[['Empresa', 'Vantagem_Lider_%']].sort_values('Vantagem_Lider_%')
cores_bar = ['#C62828' if v > 0 else '#1565C0' for v in df_vantagem['Vantagem_Lider_%']]
plt.figure(figsize=(8, 5))
plt.bar(df_vantagem['Empresa'], df_vantagem['Vantagem_Lider_%'], color=cores_bar)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Vantagem Stackelberg vs Cournot (%)\n(positivo = Stackelberg melhor)', fontsize=12)
plt.ylabel('Variação do Lucro (%)')
plt.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(file_lucros, dpi=150)
plt.show()
plt.close()

# 4.4 Gráfico interativo: Cournot vs Stackelberg (Plotly)
df_melt = df_resultado[['Empresa', 'Lucro_Cournot', 'Lucro_Stackelberg']].melt(
    id_vars='Empresa', var_name='Modelo', value_name='Lucro')
fig_inter = px.bar(df_melt, x='Empresa', y='Lucro', color='Modelo', barmode='group',
                   title='Lucros de Equilíbrio: Cournot vs Stackelberg',
                   labels={'Lucro': 'Lucro', 'Empresa': 'Empresa'},
                   color_discrete_map={'Lucro_Cournot': '#1565C0', 'Lucro_Stackelberg': '#C62828'})
fig_inter.write_html(file_interativo)
fig_inter.show()
print(f"Gráfico interativo salvo: {file_interativo}")

# 4.5 Superfície 3D: lucro do líder em função de (q_L, q_seguidor)
q_l_range  = np.linspace(0, 50, 60)
q_s_range  = np.linspace(0, 50, 60)
QL, QS     = np.meshgrid(q_l_range, q_s_range)
P_surf     = np.maximum(0, a - b * (QL + QS))
Lucro_surf = (P_surf - custo[lider]) * QL

fig_3d = go.Figure(data=[go.Surface(z=Lucro_surf, x=q_l_range, y=q_s_range,
                                    colorscale='Viridis', opacity=0.85)])
fig_3d.add_trace(go.Scatter3d(x=[q_L_st], y=[sum(q_segs_st.values())], z=[-lucro_lider(q_L_st)],
                              mode='markers',
                              marker=dict(size=8, color='red', symbol='diamond'),
                              name='Ótimo Stackelberg'))
fig_3d.update_layout(
    title='Superfície de Lucro do Líder (Stackelberg)',
    scene=dict(xaxis_title='q_Líder', yaxis_title='q_Seguidores', zaxis_title='Lucro_Líder')
)
fig_3d.write_html(file_3d)
fig_3d.show()
print(f"Gráfico 3D salvo: {file_3d}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_equilibrio}")
print(f"  {file_comparativo}")
print(f"  {file_lucros}")
print(f"  {file_interativo}")
print(f"  {file_3d}")

In [ ]:
# =============================================================
# STEP 08 - Otimização de Pareto (Fronteira Eficiente)
# -------------------------------------------------------------
# Objetivo: Construir a fronteira eficiente de Pareto para um
#           portfólio de ações via simulação de Monte Carlo.
#           Identifica carteiras ótimas no sentido de Pareto
#           (maior retorno dado o risco, ou menor risco dado
#           o retorno). Compara portfólio Sharpe máximo com
#           portfólio de mínima variância.
# Entrada  : output/base_step_00_yahoo_finance.csv  (gerado pelo step-00)
# Saídas   : output/base_step_08_pareto_portfolios.csv
#            output/base_step_08_pareto_fronteira.png
#            output/base_step_08_pareto_pesos_otimos.png
#            output/base_step_08_pareto_retornos_hist.png
#            output/base_step08_pareto_fronteira_interativa.html
#            output/base_step08_pareto_3d.html
# Depende  : STEP 00
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy.optimize import minimize

file_output          = 'output/base_step_08_pareto_portfolios.csv'
file_fronteira       = 'output/base_step_08_pareto_fronteira.png'
file_pesos           = 'output/base_step_08_pareto_pesos_otimos.png'
file_retornos_hist   = 'output/base_step_08_pareto_retornos_hist.png'
file_fronteira_inter = 'output/base_step08_pareto_fronteira_interativa.html'
file_3d              = 'output/base_step08_pareto_3d.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Aquisição de dados (base auditada step-00) ----------
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'JNJ', 'XOM', 'BRK-B']
print("Carregando base auditada do step-00 (Yahoo Finance)...")
df_cache = pd.read_csv('output/base_step_00_yahoo_finance.csv',
                       sep=';', decimal=',', encoding='utf-8-sig')
df_cache['Data'] = pd.to_datetime(df_cache['Data'])
mask = (df_cache['Ticker'].isin(tickers) &
        (df_cache['Data'] >= '2019-01-01') &
        (df_cache['Data'] <= '2023-12-31'))
dados = df_cache[mask].pivot(index='Data', columns='Ticker', values='Close')
dados = dados.dropna(axis=1, how='all')
dados.columns = [c.replace('-', '_') for c in dados.columns]
tickers_validos = list(dados.columns)
print(f"Dados carregados: {dados.shape[0]} observações, {len(tickers_validos)} ativos válidos")

# ---------- 2. Otimização de Pareto via Monte Carlo ----------
retornos    = dados.pct_change().dropna()
ret_medios  = retornos.mean() * 252
cov_matrix  = retornos.cov() * 252
n_ativos    = len(tickers_validos)
N_SIM       = 8000
np.random.seed(42)

port_ret  = np.zeros(N_SIM)
port_vol  = np.zeros(N_SIM)
port_shrp = np.zeros(N_SIM)
port_w    = np.zeros((N_SIM, n_ativos))

for i in range(N_SIM):
    w = np.random.dirichlet(np.ones(n_ativos))
    r = np.dot(w, ret_medios)
    v = np.sqrt(np.dot(w.T, np.dot(cov_matrix, w)))
    port_ret[i]  = r
    port_vol[i]  = v
    port_shrp[i] = r / v
    port_w[i]    = w

# Portfólio Sharpe Máximo
idx_sharpe = np.argmax(port_shrp)
w_sharpe   = port_w[idx_sharpe]

# Portfólio de Mínima Variância (via scipy)
def vol_portfolio(w):
    return np.sqrt(np.dot(w.T, np.dot(cov_matrix, w)))

constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
bounds      = tuple((0, 1) for _ in range(n_ativos))
res_mv      = minimize(vol_portfolio, x0=np.ones(n_ativos) / n_ativos,
                       method='SLSQP', bounds=bounds, constraints=constraints)
w_minvar    = res_mv.x
ret_minvar  = np.dot(w_minvar, ret_medios)
vol_minvar  = vol_portfolio(w_minvar)

# Identificar fronteira de Pareto (eficiente):
# Um portfólio p é Pareto-dominado se existe q com ret_q>=ret_p e vol_q<=vol_p (e ao menos uma desigualdade estrita)
is_pareto   = np.ones(N_SIM, dtype=bool)
for i in range(N_SIM):
    dominated = ((port_ret >= port_ret[i]) & (port_vol <= port_vol[i]) &
                 ((port_ret > port_ret[i]) | (port_vol < port_vol[i])))
    if dominated.any():
        is_pareto[i] = False

n_pareto = is_pareto.sum()
print(f"\nPortfólios simulados : {N_SIM}")
print(f"Portfólios Pareto    : {n_pareto}")
print(f"Sharpe máx           : {port_shrp[idx_sharpe]:.4f}  "
      f"(ret={port_ret[idx_sharpe]:.4f}, vol={port_vol[idx_sharpe]:.4f})")
print(f"Mínima variância     : ret={ret_minvar:.4f}, vol={vol_minvar:.4f}")

# ---------- 3. Salvar resultados ----------
df_portfolios = pd.DataFrame({
    'Retorno_Anual':  np.round(port_ret, 6),
    'Volatilidade':   np.round(port_vol, 6),
    'Sharpe':         np.round(port_shrp, 6),
    'Pareto_Eficiente': is_pareto.astype(int),
})
df_pesos = pd.DataFrame(port_w, columns=[f'Peso_{t}' for t in tickers_validos])
df_resultado = pd.concat([df_portfolios, df_pesos], axis=1)
df_resultado.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Fronteira Eficiente de Pareto
# 4.1 Fronteira Eficiente de Pareto
plt.figure(figsize=(10, 6))
plt.scatter(port_vol[~is_pareto] * 100, port_ret[~is_pareto] * 100,
            c=port_shrp[~is_pareto], cmap='Blues', alpha=0.3, s=8, label='Portfólios simulados')

sc = plt.scatter(port_vol[is_pareto] * 100, port_ret[is_pareto] * 100,  # ← capturar retorno
            c=port_shrp[is_pareto], cmap='YlOrRd', alpha=0.8, s=18, label='Fronteira de Pareto')

plt.scatter(port_vol[idx_sharpe] * 100, port_ret[idx_sharpe] * 100,
            marker='*', s=400, color='gold', edgecolors='black', linewidths=1, zorder=10,
            label=f'Sharpe Máx ({port_shrp[idx_sharpe]:.2f})')
plt.scatter(vol_minvar * 100, ret_minvar * 100,
            marker='D', s=200, color='cyan', edgecolors='navy', linewidths=1.5, zorder=10,
            label='Mínima Variância')

plt.colorbar(sc, label='Índice de Sharpe')  # ← usar sc em vez de ScalarMappable manual
plt.xlabel('Volatilidade Anual (%)', fontsize=11)
plt.ylabel('Retorno Anual (%)', fontsize=11)
plt.title('Fronteira Eficiente de Pareto — Otimização de Portfólio', fontsize=12)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(file_fronteira, dpi=150)
plt.show()
plt.close()

# 4.2 Pesos dos portfólios ótimos
# Substituir pie chart por barplot horizontal ordenado (SKILL: pie EVITAR com >5 categorias;
# barplot horizontal é superior para comparar proporções exatas entre muitos grupos)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cores_bar = sns.color_palette('tab10', n_ativos)

# Portfólio Sharpe Máximo
df_sh = pd.DataFrame({'Ativo': tickers_validos, 'Peso': w_sharpe * 100})
df_sh = df_sh[df_sh['Peso'] > 0.5].sort_values('Peso')   # exibir apenas pesos relevantes, ordenados
axes[0].barh(df_sh['Ativo'], df_sh['Peso'],
             color=[cores_bar[tickers_validos.index(t)] for t in df_sh['Ativo']])
axes[0].set_xlabel('Peso no Portfólio (%)')
axes[0].set_title('Portfólio — Sharpe Máximo', fontsize=11)
axes[0].grid(True, axis='x', alpha=0.4)
for i, (_, row) in enumerate(df_sh.iterrows()):
    axes[0].text(row['Peso'] + 0.3, i, f"{row['Peso']:.1f}%", va='center', fontsize=9)

# Portfólio Mínima Variância
df_mv = pd.DataFrame({'Ativo': tickers_validos, 'Peso': w_minvar * 100})
df_mv = df_mv[df_mv['Peso'] > 0.5].sort_values('Peso')
axes[1].barh(df_mv['Ativo'], df_mv['Peso'],
             color=[cores_bar[tickers_validos.index(t)] for t in df_mv['Ativo']])
axes[1].set_xlabel('Peso no Portfólio (%)')
axes[1].set_title('Portfólio — Mínima Variância', fontsize=11)
axes[1].grid(True, axis='x', alpha=0.4)
for i, (_, row) in enumerate(df_mv.iterrows()):
    axes[1].text(row['Peso'] + 0.3, i, f"{row['Peso']:.1f}%", va='center', fontsize=9)

plt.suptitle('Composição dos Portfólios Ótimos de Pareto', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_pesos, dpi=150)
plt.show()
plt.close()

# 4.3 Distribuição de retornos dos portfólios
plt.figure(figsize=(10, 5))
sns.histplot(port_ret * 100, bins=60, kde=True, color='#1565C0', label='Todos portfólios')
sns.histplot(port_ret[is_pareto] * 100, bins=40, kde=True, color='#C62828', label='Pareto Eficientes')
plt.axvline(port_ret[idx_sharpe] * 100, color='gold', linewidth=2, linestyle='--', label='Sharpe Máx')
plt.axvline(ret_minvar * 100, color='cyan', linewidth=2, linestyle='--', label='Mín Variância')
plt.xlabel('Retorno Anual (%)')
plt.ylabel('Frequência')
plt.title('Distribuição de Retornos — Portfólios Simulados vs Pareto', fontsize=12)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_retornos_hist, dpi=150)
plt.show()
plt.close()

# 4.4 Fronteira interativa (Plotly)
tipo = np.where(is_pareto, 'Pareto Eficiente', 'Dominado')
fig_inter = px.scatter(
    x=port_vol * 100, y=port_ret * 100, color=tipo,
    color_discrete_map={'Pareto Eficiente': '#E53935', 'Dominado': '#90CAF9'},
    opacity=0.5, size_max=6,
    title='Fronteira Eficiente de Pareto — Interativo',
    labels={'x': 'Volatilidade (%)', 'y': 'Retorno Anual (%)'},
)
fig_inter.add_scatter(x=[port_vol[idx_sharpe] * 100], y=[port_ret[idx_sharpe] * 100],
                      mode='markers', marker=dict(symbol='star', size=18, color='gold',
                                                   line=dict(color='black', width=1)),
                      name='Sharpe Máximo')
fig_inter.add_scatter(x=[vol_minvar * 100], y=[ret_minvar * 100],
                      mode='markers', marker=dict(symbol='diamond', size=14, color='cyan',
                                                   line=dict(color='navy', width=1.5)),
                      name='Mínima Variância')
fig_inter.write_html(file_fronteira_inter)
fig_inter.show()
print(f"Fronteira interativa salva: {file_fronteira_inter}")

# 4.5 Superfície 3D: Retorno × Volatilidade × Sharpe
fig_3d = px.scatter_3d(
    x=port_vol * 100, y=port_ret * 100, z=port_shrp,
    color=port_shrp, color_continuous_scale='Viridis',
    opacity=0.4, size_max=4,
    title='Espaço Risco-Retorno-Sharpe (3D)',
    labels={'x': 'Volatilidade (%)', 'y': 'Retorno (%)', 'z': 'Sharpe'},
)
fig_3d.write_html(file_3d)
fig_3d.show()
print(f"Gráfico 3D salvo: {file_3d}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_fronteira}")
print(f"  {file_pesos}")
print(f"  {file_retornos_hist}")
print(f"  {file_fronteira_inter}")
print(f"  {file_3d}")

In [ ]:
# =============================================================
# STEP 09 - Análise Envoltória de Dados (DEA)
# -------------------------------------------------------------
# Objetivo: Medir a eficiência relativa de países (DMUs) usando
#           DEA com modelo CRS (Constant Returns to Scale).
#           Inputs: capital (formação bruta de capital / PIB) e
#           trabalho (força de trabalho). Output: PIB per capita.
#           Identifica fronteira de eficiência e benchmarks.
# Entrada  : output/base_step_01_world_bank.csv  (gerado pelo step-01)
# Saídas   : output/base_step_09_dea_eficiencia.csv
#            output/base_step_09_dea_scores.png
#            output/base_step_09_dea_inputs_outputs.png
#            output/base_step_09_dea_benchmarks.png
#            output/base_step09_dea_interativo.html
#            output/base_step09_dea_3d.html
# Depende  : STEP 01
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy.optimize import linprog

file_output          = 'output/base_step_09_dea_eficiencia.csv'
file_scores          = 'output/base_step_09_dea_scores.png'
file_inputs_outputs  = 'output/base_step_09_dea_inputs_outputs.png'
file_benchmarks      = 'output/base_step_09_dea_benchmarks.png'
file_interativo      = 'output/base_step09_dea_interativo.html'
file_3d              = 'output/base_step09_dea_3d.html'
file_cache           = 'output/base_step_09_world_bank_raw.csv'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Aquisição de dados (base auditada step-01) ----------
# Países do DEA (subconjunto dos 33 países do step-01)
paises_dea = [
    'USA', 'CHN', 'DEU', 'JPN', 'GBR', 'FRA', 'BRA', 'IND', 'CAN', 'KOR',
    'AUS', 'ESP', 'MEX', 'IDN', 'NLD', 'CHE', 'SAU', 'TUR', 'ARG', 'ZAF',
    'NGA', 'EGY', 'POL', 'SWE', 'NOR', 'PRT', 'COL', 'CHL', 'PER', 'PHL',
]
print("Carregando base auditada do step-01 (World Bank)...")
df_wb_raw = pd.read_csv('output/base_step_01_world_bank.csv',
                        sep=';', decimal=',', encoding='utf-8-sig')
# Renomear coluna para compatibilidade com o modelo DEA deste step
df_wb_raw = df_wb_raw.rename(columns={'Capital_pct_PIB': 'Formacao_Capital_pct_PIB'})
# Filtrar países DEA e ano 2021
df_wb = (df_wb_raw[df_wb_raw['ISO'].isin(paises_dea) & (df_wb_raw['Ano'] == 2021)]
         .dropna(subset=['PIB_pc_USD', 'Formacao_Capital_pct_PIB', 'Forca_Trabalho'])
         .reset_index(drop=True))
print(f"DMUs disponíveis: {len(df_wb)} países (ano 2021)")

# Salvar cache local para dependência dos steps 13 e 14
os.makedirs('output', exist_ok=True)
df_wb.to_csv(file_cache,
             sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"Cache salvo: {file_cache}")

df = df_wb.copy()
# Ajuste de nomes para o restante do código
df = df.rename(columns={'PIB_pc_USD': 'PIB_per_capita_USD'})
for col in ['PIB_per_capita_USD', 'Formacao_Capital_pct_PIB', 'Forca_Trabalho']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna().reset_index(drop=True)

# ---------- 2. DEA — Modelo CRS (CCR) via Programação Linear ----------
# Para cada DMU k: max θ_k s.t.
#   Σ λ_j * y_j >= θ_k * y_k     (output constraint)
#   Σ λ_j * x_ij <= x_ik         (input constraints)
#   λ_j >= 0                      (intensidade)
# Reformulado como min -θ:
#   c = [-1, 0, 0, ..., 0]  (apenas θ na obj)
#   variáveis: [θ, λ_1, ..., λ_n]

# Normalização Min-Max para DEA
def minmax(s): return (s - s.min()) / (s.max() - s.min() + 1e-12) + 0.01

Y = minmax(df['PIB_per_capita_USD'].values)          # output (1D)
X1 = minmax(df['Formacao_Capital_pct_PIB'].values)   # input 1
X2 = minmax(df['Forca_Trabalho'].values)             # input 2
X  = np.vstack([X1, X2])                             # 2 x n_dmu

n_dmu = len(df)
eficiencias = np.zeros(n_dmu)

for k in range(n_dmu):
    # min c @ [θ, λ_1..λ_n]  =>  c = [-1, 0..0]
    c_obj = np.zeros(1 + n_dmu)
    c_obj[0] = -1.0   # maximizar θ

    # Restrição output: -Y @ λ + θ * Y[k] <= 0  => -Y[j]*λ[j] + θ*Y[k] <= 0
    # Ub format: A_ub @ x <= b_ub
    A_out = np.zeros((1, 1 + n_dmu))
    A_out[0, 0] = Y[k]          # coef de θ
    A_out[0, 1:] = -Y            # coef de λ
    b_out = np.array([0.0])

    # Restrição inputs: X[m,:] @ λ <= X[m,k]  para m=0,1
    A_inp = np.zeros((2, 1 + n_dmu))
    b_inp = np.zeros(2)
    for m in range(2):
        A_inp[m, 1:] = X[m, :]
        b_inp[m]     = X[m, k]

    A_ub = np.vstack([A_out, A_inp])
    b_ub = np.hstack([b_out, b_inp])

    bounds_lp = [(0, None)] * (1 + n_dmu)   # θ >= 0, λ >= 0

    res = linprog(c_obj, A_ub=A_ub, b_ub=b_ub, bounds=bounds_lp, method='highs')
    eficiencias[k] = res.x[0] if res.success else np.nan

df['Eficiencia_DEA'] = np.round(eficiencias, 6)
df['Rank']           = df['Eficiencia_DEA'].rank(ascending=False).astype(int)
df_sorted            = df.sort_values('Eficiencia_DEA', ascending=False).reset_index(drop=True)

print(f"\nTop 5 mais eficientes:")
print(df_sorted[['Pais', 'Eficiencia_DEA']].head())
print(f"\nFronteira eficiente (θ=1.0): {(df['Eficiencia_DEA'] >= 0.999).sum()} DMUs")

# ---------- 3. Salvar resultados ----------
df_sorted.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Scores de eficiência DEA por país
top_n = min(25, len(df_sorted))
df_plot = df_sorted.head(top_n)
cores_bar = ['#C62828' if e >= 0.999 else '#1565C0' for e in df_plot['Eficiencia_DEA']]
plt.figure(figsize=(12, 6))
plt.barh(df_plot['Pais'][::-1], df_plot['Eficiencia_DEA'][::-1], color=cores_bar[::-1])
plt.axvline(1.0, color='gold', linewidth=1.5, linestyle='--', label='Fronteira (θ=1)')
plt.xlabel('Score de Eficiência DEA (CRS)', fontsize=11)
plt.title(f'Scores de Eficiência DEA — Top {top_n} países (2021)', fontsize=12)
plt.legend(fontsize=9)
plt.grid(True, axis='x', alpha=0.35)
plt.tight_layout()
plt.savefig(file_scores, dpi=150)
plt.show()
plt.close()

# 4.2 Inputs × Output por país
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(df['Formacao_Capital_pct_PIB'], df['PIB_per_capita_USD'],
                     c=df['Eficiencia_DEA'], cmap='coolwarm', s=60, edgecolors='grey', linewidths=0.4)
plt.colorbar(sc, ax=axes[0], label='Eficiência DEA')
axes[0].set_xlabel('Formação de Capital (% PIB)')
axes[0].set_ylabel('PIB per capita (USD)')
axes[0].set_title('Capital vs PIB per capita')
axes[0].grid(True, alpha=0.3)

sc2 = axes[1].scatter(np.log1p(df['Forca_Trabalho']), df['PIB_per_capita_USD'],
                      c=df['Eficiencia_DEA'], cmap='coolwarm', s=60, edgecolors='grey', linewidths=0.4)
plt.colorbar(sc2, ax=axes[1], label='Eficiência DEA')
axes[1].set_xlabel('log(Força de Trabalho)')
axes[1].set_ylabel('PIB per capita (USD)')
axes[1].set_title('Trabalho (log) vs PIB per capita')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Inputs × Output — Mapa de Eficiência DEA', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_inputs_outputs, dpi=150)
plt.show()
plt.close()

# 4.3 Distribuição dos scores de eficiência
plt.figure(figsize=(9, 5))
sns.histplot(df['Eficiencia_DEA'], bins=20, kde=True, color='#1565C0')
plt.axvline(df['Eficiencia_DEA'].mean(), color='red', linestyle='--',
            label=f'Média: {df["Eficiencia_DEA"].mean():.3f}')
plt.axvline(1.0, color='gold', linestyle='--', linewidth=1.5, label='Fronteira (θ=1)')
plt.xlabel('Score de Eficiência DEA')
plt.ylabel('Frequência')
plt.title('Distribuição dos Scores de Eficiência DEA (CRS)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_benchmarks, dpi=150)
plt.show()
plt.close()

# 4.4 Gráfico interativo: eficiência por país (Plotly)
fig_inter = px.bar(df_sorted.head(top_n), x='Eficiencia_DEA', y='Pais', orientation='h',
                   color='Eficiencia_DEA', color_continuous_scale='RdBu',
                   title=f'Scores de Eficiência DEA — Top {top_n} Países',
                   labels={'Eficiencia_DEA': 'Eficiência DEA', 'Pais': 'País'})
fig_inter.add_vline(x=1.0, line_dash='dash', line_color='gold', annotation_text='Fronteira')
fig_inter.write_html(file_interativo)
fig_inter.show()
print(f"Gráfico interativo salvo: {file_interativo}")

# 4.5 Gráfico 3D: Capital × Trabalho × PIB, colorido por eficiência
fig_3d = px.scatter_3d(
    df, x='Formacao_Capital_pct_PIB', y=np.log1p(df['Forca_Trabalho']),
    z='PIB_per_capita_USD', color='Eficiencia_DEA',
    color_continuous_scale='RdBu', hover_name='Pais',
    title='DEA 3D: Capital × Trabalho × PIB per capita',
    labels={'x': 'Capital (% PIB)', 'y': 'log(Trabalho)', 'z': 'PIB per capita'},
    opacity=0.85
)
fig_3d.write_html(file_3d)
fig_3d.show()
print(f"Gráfico 3D salvo: {file_3d}")

print("\nArquivos salvos:")
print(f"  {file_cache}")
print(f"  {file_output}")
print(f"  {file_scores}")
print(f"  {file_inputs_outputs}")
print(f"  {file_benchmarks}")
print(f"  {file_interativo}")
print(f"  {file_3d}")

In [ ]:
# =============================================================
# STEP 10 - Análise de Dados Topológica (TDA)
# -------------------------------------------------------------
# Objetivo: Extrair padrões estruturais latentes nos dados de
#           transações do Online Retail II via Homologia
#           Persistente. Computa features RFM (Recência,
#           Frequência, Monetário) por cliente, aplica ripser
#           para calcular diagramas de persistência (H0, H1) e
#           extrai números de Betti para segmentação topológica.
# Entrada  : output/base_step_02_online_retail.csv  (gerado pelo step-02)
# Saídas   : output/base_step_10_tda_features_rfm.csv
#            output/base_step_10_tda_persistencia.png
#            output/base_step_10_tda_betti.png
#            output/base_step_10_tda_rfm_distribuicao.png
#            output/base_step10_tda_scatter_interativo.html
#            output/base_step10_tda_rfm_3d.html
# Depende  : STEP 02
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from sklearn.preprocessing import StandardScaler
from datetime import datetime

file_output         = 'output/base_step_10_tda_features_rfm.csv'
file_persistencia   = 'output/base_step_10_tda_persistencia.png'
file_betti          = 'output/base_step_10_tda_betti.png'
file_rfm_dist       = 'output/base_step_10_tda_rfm_distribuicao.png'
file_scatter_inter  = 'output/base_step10_tda_scatter_interativo.html'
file_rfm_3d         = 'output/base_step10_tda_rfm_3d.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Aquisição de dados (base auditada step-02) ----------
print("Carregando base auditada do step-02 (Online Retail II UCI)...")
df_raw = pd.read_csv('output/base_step_02_online_retail.csv',
                     sep=';', decimal=',', encoding='utf-8-sig', low_memory=False)
print(f"Dados carregados: {len(df_raw)} linhas")

# ---------- 2. Preparação RFM e Features Topológicas ----------
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
# Normalização defensiva: Online Retail II pode salvar 'InvoiceDate' ou 'Invoice Date'
if 'InvoiceDate' in df.columns and 'Invoice Date' not in df.columns:
    df = df.rename(columns={'InvoiceDate': 'Invoice Date'})

# Limpar dados
df = df[df['Customer ID'].notna()].copy()
df = df[df['Quantity'] > 0].copy()
df = df[df['Price'] > 0].copy()
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'])
df['Revenue']      = df['Quantity'] * df['Price']
df['Customer ID']  = df['Customer ID'].astype(int)

data_ref = df['Invoice Date'].max()

# Calcular RFM por cliente
rfm = df.groupby('Customer ID').agg(
    Recencia    = ('Invoice Date', lambda x: (data_ref - x.max()).days),
    Frequencia  = ('Invoice', 'nunique'),
    Monetario   = ('Revenue', 'sum'),
).reset_index()
rfm = rfm[rfm['Monetario'] > 0].reset_index(drop=True)
print(f"\nClientes com RFM: {len(rfm)}")

# Normalizar para TDA
scaler = StandardScaler()
X_rfm  = scaler.fit_transform(rfm[['Recencia', 'Frequencia', 'Monetario']].values)

# Amostra para homologia persistente (ripser pode ser lento em amostras grandes)
np.random.seed(42)
n_sample = min(500, len(X_rfm))
idx_sample = np.random.choice(len(X_rfm), n_sample, replace=False)
X_sample   = X_rfm[idx_sample]

# Calcular Homologia Persistente
try:
    from ripser import ripser
    from persim import plot_diagrams
    diagramas = ripser(X_sample, maxdim=1)['dgms']
    TDA_DISPONIVEL = True
    print(f"Ripser disponível. Calculando H0 e H1 em {n_sample} pontos...")
except ImportError:
    TDA_DISPONIVEL = False
    print("ripser não instalado. Usando aproximação via distâncias para visualização.")
    # Fallback: Rips filtration manual via matriz de distâncias
    from sklearn.metrics import pairwise_distances
    D = pairwise_distances(X_sample)
    thresholds = np.linspace(0, D.max() * 0.6, 50)
    # Simular diagrama H0 (componentes conectados)
    betti_0 = []
    for eps in thresholds:
        adj = (D < eps).astype(int) - np.eye(n_sample)
        # ncomp = n_sample - adj.sum(axis=1).clip(0,1).sum() // 2  # aproximado
        betti_0.append(max(1, n_sample - int((D < eps).sum() / 2)))
    diagramas = None

# Números de Betti estimados (componentes em H0, loops em H1)
if TDA_DISPONIVEL:
    dgm_H0 = diagramas[0]
    dgm_H1 = diagramas[1]
    # Filtrar nascimentos/mortes infinitas
    dgm_H0_fin = dgm_H0[np.isfinite(dgm_H0[:, 1])]
    dgm_H1_fin = dgm_H1[np.isfinite(dgm_H1[:, 1])] if len(dgm_H1) > 0 else np.empty((0, 2))
    persistencia_H0 = dgm_H0_fin[:, 1] - dgm_H0_fin[:, 0]
    persistencia_H1 = dgm_H1_fin[:, 1] - dgm_H1_fin[:, 0] if len(dgm_H1_fin) > 0 else np.array([])
    betti_0_est = (persistencia_H0 > persistencia_H0.mean()).sum() if len(persistencia_H0) > 0 else 1
    betti_1_est = (persistencia_H1 > 0.1).sum() if len(persistencia_H1) > 0 else 0
else:
    betti_0_est, betti_1_est = 3, 1  # estimativa fallback

# Segmentação RFM simples (quartis) para coloração
rfm['Segmento_R'] = pd.qcut(rfm['Recencia'],   q=4, labels=['R4','R3','R2','R1'])
rfm['Segmento_F'] = pd.qcut(rfm['Frequencia'].rank(method='first'), q=4, labels=['F1','F2','F3','F4'])
rfm['Segmento_M'] = pd.qcut(rfm['Monetario'],  q=4, labels=['M1','M2','M3','M4'])
rfm['RFM_Score']  = (rfm['Segmento_R'].str[1].astype(int) +
                     rfm['Segmento_F'].str[1].astype(int) +
                     rfm['Segmento_M'].str[1].astype(int))
rfm['Cluster']    = pd.cut(rfm['RFM_Score'], bins=[2, 5, 8, 12],
                           labels=['Bronze', 'Prata', 'Ouro'], include_lowest=True)
rfm['Betti_0']    = betti_0_est
rfm['Betti_1']    = betti_1_est

print(f"\nNúmeros de Betti estimados — H0: {betti_0_est}, H1: {betti_1_est}")

# ---------- 3. Salvar resultados ----------
rfm.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Diagrama de Persistência (H0 e H1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
if TDA_DISPONIVEL and diagramas is not None:
    # H0
    if len(dgm_H0_fin) > 0:
        axes[0].scatter(dgm_H0_fin[:, 0], dgm_H0_fin[:, 1], c='#1565C0', s=20, alpha=0.7, label='H0')
    lim0 = max(dgm_H0_fin.max() if len(dgm_H0_fin) > 0 else 1, 0.1)
    axes[0].plot([0, lim0], [0, lim0], 'k--', linewidth=0.8)
    axes[0].set_xlabel('Nascimento')
    axes[0].set_ylabel('Morte')
    axes[0].set_title('Diagrama de Persistência — H0 (Componentes)')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)
    # H1
    if len(dgm_H1_fin) > 0:
        axes[1].scatter(dgm_H1_fin[:, 0], dgm_H1_fin[:, 1], c='#C62828', s=30, alpha=0.8, label='H1')
    lim1 = max(dgm_H1_fin.max() if len(dgm_H1_fin) > 0 else 1, 0.1)
    axes[1].plot([0, lim1], [0, lim1], 'k--', linewidth=0.8)
    axes[1].set_xlabel('Nascimento')
    axes[1].set_ylabel('Morte')
    axes[1].set_title('Diagrama de Persistência — H1 (Loops/Ciclos)')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'Instale ripser para diagrama completo\npip install ripser persim',
                 ha='center', va='center', fontsize=12, transform=axes[0].transAxes)
    axes[0].set_title('Diagrama H0 — ripser não disponível')
    axes[1].text(0.5, 0.5, 'Números de Betti estimados:\nH0 = ' + str(betti_0_est) +
                 '  |  H1 = ' + str(betti_1_est),
                 ha='center', va='center', fontsize=13, transform=axes[1].transAxes)
    axes[1].set_title('Diagrama H1 — estimativa')
plt.suptitle('Análise de Dados Topológica (TDA) — Online Retail II', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_persistencia, dpi=150)
plt.show()
plt.close()

# 4.2 Números de Betti e persistência por dimensão
# Lollipop chart: preferível ao barplot quando poucas barras com alturas possivelmente similares
# (evita Moiré effect e é mais limpo com n=2 grupos)
betti_vals = pd.DataFrame({'Dimensão': ['H0 (Componentes)', 'H1 (Loops/Ciclos)'],
                           'Betti': [betti_0_est, betti_1_est]})
cores_betti = ['#1565C0', '#C62828']
plt.figure(figsize=(7, 4))
plt.hlines(betti_vals['Dimensão'], xmin=0, xmax=betti_vals['Betti'],
           color='#b0b0b0', linewidth=2)
for i, (_, row) in enumerate(betti_vals.iterrows()):
    plt.plot(row['Betti'], row['Dimensão'], 'o', color=cores_betti[i], markersize=14, zorder=5)
    plt.text(row['Betti'] + 0.05, row['Dimensão'], str(int(row['Betti'])),
             va='center', fontsize=13, fontweight='bold', color=cores_betti[i])
plt.title('Números de Betti por Dimensão Homológica', fontsize=12)
plt.xlabel('Número de Betti (β)')
plt.xlim(left=0)
plt.grid(True, axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig(file_betti, dpi=150)
plt.show()
plt.close()

# 4.3 Distribuição RFM por segmento (Cluster)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, cor, titulo in zip(axes,
    ['Recencia', 'Frequencia', 'Monetario'],
    ['#1565C0', '#2E7D32', '#C62828'],
    ['Recência (dias)', 'Frequência (pedidos)', 'Monetário (R$)']):
    sns.histplot(data=rfm, x=col, hue='Cluster', bins=30, kde=True, ax=ax, legend=ax == axes[2])
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)
plt.suptitle('Distribuição RFM por Segmento Topológico (Cluster)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_rfm_dist, dpi=150)
plt.show()
plt.close()

# 4.4 Scatter interativo RFM (Plotly)
rfm_plot = rfm.sample(min(2000, len(rfm)), random_state=42)
fig_scat = px.scatter(rfm_plot, x='Recencia', y='Monetario', color='Cluster',
                      size=np.log1p(rfm_plot['Frequencia']) * 3 + 2,
                      hover_data=['Customer ID', 'Frequencia', 'RFM_Score'],
                      title='TDA — Segmentação RFM: Recência × Monetário',
                      color_discrete_map={'Bronze': '#CD7F32', 'Prata': '#A8A9AD', 'Ouro': '#FFD700'})
fig_scat.write_html(file_scatter_inter)
fig_scat.show()
print(f"Scatter interativo salvo: {file_scatter_inter}")

# 4.5 Espaço RFM 3D (Plotly)
fig_3d = px.scatter_3d(rfm_plot, x='Recencia', y='Frequencia', z='Monetario', color='Cluster',
                        hover_name='Customer ID', opacity=0.7,
                        color_discrete_map={'Bronze': '#CD7F32', 'Prata': '#A8A9AD', 'Ouro': '#FFD700'},
                        title='Espaço Topológico RFM 3D — TDA Online Retail II')
fig_3d.write_html(file_rfm_3d)
fig_3d.show()
print(f"Gráfico 3D RFM salvo: {file_rfm_3d}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_persistencia}")
print(f"  {file_betti}")
print(f"  {file_rfm_dist}")
print(f"  {file_scatter_inter}")
print(f"  {file_rfm_3d}")

In [ ]:
# =============================================================
# STEP 11 - Teorema de Aproximação de Blackwell
# -------------------------------------------------------------
# Objetivo: Demonstrar o Teorema de Aproximação de Blackwell em
#           um jogo sequencial de recomendação de produtos.
#           O agente escolhe ações (categorias de produto) e
#           recebe respostas adversariais (rejeição/aceitação).
#           O algoritmo garante que o payoff médio converge ao
#           conjunto-alvo C. Mede o arrependimento externo e
#           interno ao longo das rodadas.
# Entrada  : output/base_step_10_online_retail_raw.csv
# Saídas   : output/base_step_11_blackwell_convergencia.csv
#            output/base_step_11_blackwell_regret.png
#            output/base_step_11_blackwell_trajetoria.png
#            output/base_step_11_blackwell_payoffs.png
#            output/base_step11_blackwell_convergencia_interativa.html
#            output/base_step11_blackwell_regret_interativo.html
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

file_input              = 'output/base_step_02_online_retail.csv'
file_output             = 'output/base_step_11_blackwell_convergencia.csv'
file_regret             = 'output/base_step_11_blackwell_regret.png'
file_trajetoria         = 'output/base_step_11_blackwell_trajetoria.png'
file_payoffs            = 'output/base_step_11_blackwell_payoffs.png'
file_convergencia_inter = 'output/base_step11_blackwell_convergencia_interativa.html'
file_regret_inter       = 'output/base_step11_blackwell_regret_interativo.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Carregar dados ----------
df_raw = pd.read_csv(file_input, sep=';', encoding='utf-8-sig', decimal=',')
print(f"Linhas originais: {len(df_raw)}")

df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
df = df[df['Quantity'].astype(float) > 0].copy()
df = df[df['Price'].astype(float) > 0].copy()
df['Revenue'] = df['Quantity'].astype(float) * df['Price'].astype(float)

# Extrair top categorias (primeiros 4 chars do StockCode como proxy de categoria)
df['Categoria'] = df['StockCode'].astype(str).str[:2]
top_cats = df.groupby('Categoria')['Revenue'].sum().nlargest(6).index.tolist()
df_cats  = df[df['Categoria'].isin(top_cats)].copy()
print(f"Top categorias usadas: {top_cats}")
print(f"Transações nas categorias: {len(df_cats)}")

# ---------- 2. Blackwell Approachability ----------
# Configuração do jogo:
#   Ações do agente (recomendador):   K categorias de produto
#   Respostas do ambiente (adversário): aceitar (1) ou rejeitar (0)
#   Payoff: vetor 2D (receita normalizada, taxa de aceitação)
#   Conjunto-alvo C: {(r, a) | r >= r_min, a >= a_min}  (cone positivo)

np.random.seed(2024)
K = len(top_cats)   # número de ações
T = 1000            # número de rodadas

# Estimar payoff médio e variação por categoria dos dados reais
stats_cat = df_cats.groupby('Categoria').agg(
    rev_med = ('Revenue', 'mean'),
    qtd_med = ('Quantity', 'mean')
).reindex(top_cats).fillna(0)

rev_norm = (stats_cat['rev_med'] / stats_cat['rev_med'].max()).values  # payoff dim 1
acc_norm = (stats_cat['qtd_med'] / stats_cat['qtd_med'].max()).values  # payoff dim 2

# Conjunto-alvo C: ponto de referência (média das melhores ações)
c_ref = np.array([rev_norm.mean(), acc_norm.mean()])

# Algoritmo de Blackwell:
# Distribuição mista p_t sobre K ações
# Atualização: p_{t+1} proporcional à projeção do payoff médio em C
payoff_medio  = np.zeros(2)
distancia_C   = np.zeros(T)
regret_externo = np.zeros(T)
payoffs_hist  = np.zeros((T, 2))
acoes_hist    = np.zeros(T, dtype=int)
p             = np.ones(K) / K   # uniforme inicial

for t in range(T):
    # Agente escolhe ação segundo distribuição mista p
    acao = np.random.choice(K, p=p)
    acoes_hist[t] = acao

    # Adversário escolhe resposta (ruído realístico baseado em dados)
    ruido = np.random.normal(0, 0.05, 2)
    payoff_t = np.array([rev_norm[acao], acc_norm[acao]]) + ruido
    payoff_t = np.clip(payoff_t, 0, 1)
    payoffs_hist[t] = payoff_t

    # Atualizar payoff médio
    payoff_medio = (payoff_medio * t + payoff_t) / (t + 1)

    # Distância ao conjunto-alvo C (projeção no ponto mais próximo acima de c_ref)
    proj = np.maximum(payoff_medio, c_ref)
    distancia_C[t] = np.linalg.norm(payoff_medio - proj)

    # Arrependimento externo: diferença para a melhor ação fixa
    melhor_acao_payoff = max(rev_norm[k] + acc_norm[k] for k in range(K)) / 2
    regret_externo[t]  = melhor_acao_payoff - (payoff_medio[0] + payoff_medio[1]) / 2

    # Atualizar distribuição: direção de Blackwell
    direcao = c_ref - payoff_medio
    if np.linalg.norm(direcao) > 1e-9:
        # Escolha a ação cujo payoff maximiza o produto interno com a direção
        scores = np.array([np.dot(direcao, [rev_norm[k], acc_norm[k]]) for k in range(K)])
        scores = scores - scores.min()
        s_sum = scores.sum()
        p = scores / s_sum if s_sum > 1e-12 else np.ones(K) / K
    else:
        p = np.ones(K) / K
    # Garantir soma exatamente 1.0 (precisão float)
    p = p / p.sum()

print(f"\nDistância final ao conjunto C: {distancia_C[-1]:.6f}")
print(f"Arrependimento externo final:   {regret_externo[-1]:.6f}")
print(f"Payoff médio convergido:        R={payoff_medio[0]:.4f}, A={payoff_medio[1]:.4f}")
print(f"Ponto de referência C:          R={c_ref[0]:.4f}, A={c_ref[1]:.4f}")

# Frequência de ações
freq_acoes = pd.Series(acoes_hist).value_counts().sort_index()
print(f"\nFrequência de ações (categoria):")
for i, cat in enumerate(top_cats):
    print(f"  {cat}: {freq_acoes.get(i, 0)} vezes ({freq_acoes.get(i,0)/T*100:.1f}%)")

# ---------- 3. Salvar resultados ----------
df_resultado = pd.DataFrame({
    'Rodada':           range(1, T + 1),
    'Acao':             [top_cats[a] for a in acoes_hist],
    'Payoff_Receita':   np.round(payoffs_hist[:, 0], 6),
    'Payoff_Aceitacao': np.round(payoffs_hist[:, 1], 6),
    'Payoff_Medio_R':   np.round(np.cumsum(payoffs_hist[:, 0]) / (np.arange(T) + 1), 6),
    'Payoff_Medio_A':   np.round(np.cumsum(payoffs_hist[:, 1]) / (np.arange(T) + 1), 6),
    'Distancia_C':      np.round(distancia_C, 6),
    'Regret_Externo':   np.round(regret_externo, 6),
})
df_resultado.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Arrependimento externo ao longo das rodadas
rodadas = np.arange(1, T + 1)
bound_teorico = 1 / np.sqrt(rodadas)   # O(1/√T)

plt.figure(figsize=(11, 5))
plt.plot(rodadas, regret_externo, color='#1565C0', linewidth=1.2, label='Regret Externo (real)', alpha=0.8)
plt.plot(rodadas, bound_teorico, color='#C62828', linewidth=1.5, linestyle='--',
         label='Limite teórico O(1/√T)')
plt.axhline(0, color='black', linewidth=0.7)
plt.xlabel('Rodada (t)', fontsize=11)
plt.ylabel('Arrependimento Externo')
plt.title('Convergência do Arrependimento — Blackwell Approachability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_regret, dpi=150)
plt.show()
plt.close()

# 4.2 Trajetória do payoff médio convergindo ao conjunto C
plt.figure(figsize=(8, 7))
pm_R = df_resultado['Payoff_Medio_R'].values
pm_A = df_resultado['Payoff_Medio_A'].values
plt.plot(pm_R, pm_A, color='#1565C0', linewidth=1, alpha=0.7, label='Trajetória payoff médio')
plt.scatter(pm_R[0], pm_A[0], s=150, color='#2E7D32', zorder=5, label='Início')
plt.scatter(pm_R[-1], pm_A[-1], s=150, color='#C62828', marker='*', zorder=6, label='Final')
plt.scatter(c_ref[0], c_ref[1], s=200, color='gold', marker='D', edgecolors='black',
            linewidths=1.2, zorder=7, label='Ponto alvo C')
plt.axhline(c_ref[1], color='grey', linewidth=0.8, linestyle=':', alpha=0.6)
plt.axvline(c_ref[0], color='grey', linewidth=0.8, linestyle=':', alpha=0.6)
plt.fill_between([c_ref[0], 1.1], c_ref[1], 1.1, alpha=0.08, color='green', label='Conjunto C')
plt.xlabel('Payoff Médio — Receita', fontsize=11)
plt.ylabel('Payoff Médio — Aceitação', fontsize=11)
plt.title('Trajetória de Convergência ao Conjunto-Alvo C', fontsize=12)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.35)
plt.xlim(0, 1.05)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig(file_trajetoria, dpi=150)
plt.show()
plt.close()

# 4.3 Frequência de ações e distância ao conjunto C
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(top_cats, [freq_acoes.get(i, 0) / T * 100 for i in range(K)], color='#1565C0')
axes[0].set_xlabel('Categoria de Produto')
axes[0].set_ylabel('Frequência (%)')
axes[0].set_title('Distribuição de Ações — Blackwell')
axes[0].grid(True, axis='y', alpha=0.4)

axes[1].plot(rodadas, distancia_C, color='#C62828', linewidth=1.2)
axes[1].plot(rodadas, bound_teorico * 0.5, color='#FF9800', linewidth=1.5,
             linestyle='--', label='O(1/√T) / 2')
axes[1].set_xlabel('Rodada (t)')
axes[1].set_ylabel('Distância ao Conjunto C')
axes[1].set_title('Distância ao Conjunto-Alvo C ao Longo do Tempo')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.35)
plt.suptitle('Blackwell Approachability — Diagnóstico', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_payoffs, dpi=150)
plt.show()
plt.close()

# 4.4 Convergência interativa (Plotly)
fig_inter = go.Figure()
fig_inter.add_trace(go.Scatter(x=rodadas.tolist(), y=regret_externo.tolist(),
                               mode='lines', name='Regret Externo',
                               line=dict(color='#1565C0', width=1.5)))
fig_inter.add_trace(go.Scatter(x=rodadas.tolist(), y=bound_teorico.tolist(),
                               mode='lines', name='Limite O(1/√T)',
                               line=dict(color='#C62828', width=2, dash='dash')))
fig_inter.add_trace(go.Scatter(x=rodadas.tolist(), y=distancia_C.tolist(),
                               mode='lines', name='Distância a C',
                               line=dict(color='#2E7D32', width=1.5)))
fig_inter.update_layout(title='Blackwell Approachability — Convergência Interativa',
                        xaxis_title='Rodada', yaxis_title='Valor')
fig_inter.write_html(file_convergencia_inter)
fig_inter.show()
print(f"Convergência interativa salva: {file_convergencia_inter}")

# 4.5 Trajetória interativa no espaço de payoff
fig_reg = go.Figure()
fig_reg.add_trace(go.Scatter(x=pm_R.tolist(), y=pm_A.tolist(), mode='lines+markers',
                             marker=dict(size=3, color=np.arange(T), colorscale='Blues'),
                             name='Trajetória payoff médio',
                             line=dict(color='#1565C0', width=1)))
fig_reg.add_trace(go.Scatter(x=[c_ref[0]], y=[c_ref[1]], mode='markers',
                             marker=dict(size=15, symbol='diamond', color='gold',
                                         line=dict(color='black', width=2)),
                             name='Conjunto-alvo C'))
fig_reg.update_layout(title='Trajetória de Convergência ao Conjunto C — Interativo',
                      xaxis_title='Payoff Médio (Receita)', yaxis_title='Payoff Médio (Aceitação)')
fig_reg.write_html(file_regret_inter)
fig_reg.show()
print(f"Trajetória interativa salva: {file_regret_inter}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_regret}")
print(f"  {file_trajetoria}")
print(f"  {file_payoffs}")
print(f"  {file_convergencia_inter}")
print(f"  {file_regret_inter}")

In [ ]:
# =============================================================
# STEP 12 - Teoria da Decisão Bayesiana
# -------------------------------------------------------------
# Objetivo: Aplicar inferência bayesiana para estimar o Customer
#           Lifetime Value (CLV) de clientes do Online Retail II.
#           Usa modelo Beta-Binomial para a probabilidade de
#           recompra (prior conjugado) e atualiza posteriors
#           com dados observados. Gera regras de decisão ótimas
#           para priorização de clientes.
# Entrada  : output/base_step_10_online_retail_raw.csv
# Saídas   : output/base_step_12_bayesiana_clv.csv
#            output/base_step_12_bayesiana_posterior.png
#            output/base_step_12_bayesiana_clv_dist.png
#            output/base_step_12_bayesiana_atualizacao.png
#            output/base_step12_bayesiana_interativo.html
#            output/base_step12_bayesiana_clv_3d.html
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy.stats import beta as beta_dist, gamma as gamma_dist

file_input          = 'output/base_step_02_online_retail.csv'
file_output         = 'output/base_step_12_bayesiana_clv.csv'
file_posterior      = 'output/base_step_12_bayesiana_posterior.png'
file_clv_dist       = 'output/base_step_12_bayesiana_clv_dist.png'
file_atualizacao    = 'output/base_step_12_bayesiana_atualizacao.png'
file_interativo     = 'output/base_step12_bayesiana_interativo.html'
file_clv_3d         = 'output/base_step12_bayesiana_clv_3d.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Carregar dados ----------
df_raw = pd.read_csv(file_input, sep=';', encoding='utf-8-sig', decimal=',')
print(f"Linhas originais: {len(df_raw)}")

df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
# Normalização defensiva: Online Retail II pode salvar 'InvoiceDate' ou 'Invoice Date'
if 'InvoiceDate' in df.columns and 'Invoice Date' not in df.columns:
    df = df.rename(columns={'InvoiceDate': 'Invoice Date'})
df = df[df['Customer ID'].notna()].copy()
df = df[df['Quantity'].astype(float) > 0].copy()
df = df[df['Price'].astype(float) > 0].copy()
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'])
df['Revenue']      = df['Quantity'].astype(float) * df['Price'].astype(float)
df['Customer ID']  = df['Customer ID'].astype(int)
df = df.sort_values('Invoice Date')

# ---------- 2. Inferência Bayesiana — Modelo Beta-Binomial para CLV ----------
# Dividir em janela de calibração (70%) e holdout (30%) por data
data_split = df['Invoice Date'].quantile(0.70)
df_cal     = df[df['Invoice Date'] <= data_split]
df_hold    = df[df['Invoice Date'] > data_split]

data_ref = df['Invoice Date'].max()

# Features por cliente na calibração
def calcular_rfm(df_subset, data_referencia):
    return df_subset.groupby('Customer ID').agg(
        n_pedidos    = ('Invoice', 'nunique'),
        recencia     = ('Invoice Date', lambda x: (data_referencia - x.max()).days),
        ticket_medio = ('Revenue', lambda x: x.sum() / df_subset.loc[x.index, 'Invoice'].nunique()),
        receita_tot  = ('Revenue', 'sum'),
    ).reset_index()

rfm_cal  = calcular_rfm(df_cal, data_split)
rfm_hold = calcular_rfm(df_hold, data_ref)
rfm_hold = rfm_hold.rename(columns={'n_pedidos': 'n_pedidos_hold', 'receita_tot': 'receita_hold'})

# Clientes presentes em ambas as janelas
clientes_comuns = set(rfm_cal['Customer ID']) & set(rfm_hold['Customer ID'])
rfm_cal['Recomprou'] = rfm_cal['Customer ID'].isin(clientes_comuns).astype(int)
print(f"\nClientes na calibração : {len(rfm_cal)}")
print(f"Taxa de recompra real  : {rfm_cal['Recomprou'].mean():.3f}")

# Prior Beta(α₀, β₀): crença inicial sobre probabilidade de recompra
# α₀ = 2, β₀ = 5 → prior fraco (média = 0.29, equivalente a 7 observações)
alpha0, beta0 = 2.0, 5.0

# Posterior por cliente: Beta(α₀ + s, β₀ + (n - s))
# onde s = recomprou (1 ou 0), n = 1 (observação binária por cliente)
rfm_cal['alpha_post'] = alpha0 + rfm_cal['Recomprou']
rfm_cal['beta_post']  = beta0  + (1 - rfm_cal['Recomprou'])
rfm_cal['p_recompra'] = rfm_cal['alpha_post'] / (rfm_cal['alpha_post'] + rfm_cal['beta_post'])
rfm_cal['p_recompra_std'] = np.sqrt(
    rfm_cal['alpha_post'] * rfm_cal['beta_post'] /
    ((rfm_cal['alpha_post'] + rfm_cal['beta_post'])**2 *
     (rfm_cal['alpha_post'] + rfm_cal['beta_post'] + 1))
)

# CLV Bayesiano: E[CLV] = p_recompra * ticket_medio * horizonte_esperado
# horizonte_esperado ~ Gamma prior (forma de vida do cliente)
HORIZONTE_MESES = 12
rfm_cal['CLV_Bayesiano']   = rfm_cal['p_recompra'] * rfm_cal['ticket_medio'] * rfm_cal['n_pedidos'] * (HORIZONTE_MESES / 12)
rfm_cal['CLV_Incerteza']   = rfm_cal['p_recompra_std'] * rfm_cal['ticket_medio'] * rfm_cal['n_pedidos']

# Segmentação por CLV
rfm_cal['Segmento_CLV'] = pd.qcut(rfm_cal['CLV_Bayesiano'],
                                   q=4, labels=['Bronze', 'Prata', 'Ouro', 'Diamante'])

# Prioridade de ação: taxa Bayesiana * receita esperada
rfm_cal['Prioridade'] = rfm_cal['p_recompra'] * rfm_cal['CLV_Bayesiano']
rfm_cal['Rank_CLV']   = rfm_cal['CLV_Bayesiano'].rank(ascending=False).astype(int)

print(f"\nCLV Bayesiano médio   : R$ {rfm_cal['CLV_Bayesiano'].mean():.2f}")
print(f"CLV Bayesiano mediano : R$ {rfm_cal['CLV_Bayesiano'].median():.2f}")
print(f"Top 10% CLV médio     : R$ {rfm_cal['CLV_Bayesiano'].quantile(0.9):.2f}")

# ---------- 3. Salvar resultados ----------
rfm_cal.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Distribuições Prior e Posterior por segmento
p_vals = np.linspace(0, 1, 200)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
segmentos = ['Bronze', 'Prata', 'Ouro', 'Diamante']
cores_seg  = ['#CD7F32', '#A8A9AD', '#FFD700', '#0D47A1']
for ax, seg, cor in zip(axes.flat, segmentos, cores_seg):
    sub = rfm_cal[rfm_cal['Segmento_CLV'] == seg]
    a_med, b_med = sub['alpha_post'].mean(), sub['beta_post'].mean()
    prior_vals   = beta_dist.pdf(p_vals, alpha0, beta0)
    post_vals    = beta_dist.pdf(p_vals, a_med, b_med)
    ax.plot(p_vals, prior_vals, color='grey', linestyle='--', linewidth=1.5, label='Prior')
    ax.plot(p_vals, post_vals, color=cor, linewidth=2, label=f'Posterior {seg}')
    ax.axvline(a_med / (a_med + b_med), color=cor, linestyle=':', linewidth=1.2)
    ax.set_title(f'Segmento {seg} (n={len(sub)})', fontsize=10)
    ax.set_xlabel('p (prob. recompra)')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle('Prior → Posterior Beta-Binomial por Segmento de CLV', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_posterior, dpi=150)
plt.show()
plt.close()

# 4.2 Distribuição do CLV Bayesiano por segmento
plt.figure(figsize=(11, 5))
for seg, cor in zip(segmentos, cores_seg):
    sub = rfm_cal[rfm_cal['Segmento_CLV'] == seg]
    sns.kdeplot(sub['CLV_Bayesiano'], label=f'{seg} (n={len(sub)})', color=cor, linewidth=2)
plt.axvline(rfm_cal['CLV_Bayesiano'].mean(), color='black', linestyle='--',
            linewidth=1.5, label=f'Média geral: R${rfm_cal["CLV_Bayesiano"].mean():.0f}')
plt.xlabel('CLV Bayesiano (R$)', fontsize=11)
plt.ylabel('Densidade')
plt.title('Distribuição do CLV Bayesiano por Segmento', fontsize=12)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_clv_dist, dpi=150)
plt.show()
plt.close()

# 4.3 Atualização Bayesiana: evidência acumulada
n_obs_range = np.arange(0, 21)
# Reduzido de 5 para 4 séries — SKILL: >4 linhas = spaghetti; usar ≤4 séries representativas
# Remove taxa=0.5 (central, redundante) para preservar legibilidade sem perder a mensagem analítica
recompra_rates = [0, 0.33, 0.67, 1.0]   # 4 taxas que cobrem todo o espaço de 0% a 100%
cores_rates = ['#C62828', '#FF9800', '#2E7D32', '#1565C0']
fig, ax = plt.subplots(figsize=(10, 5))
for taxa, cor in zip(recompra_rates, cores_rates):
    p_posts = []
    for n in n_obs_range:
        s   = int(round(n * taxa))
        a_n = alpha0 + s
        b_n = beta0 + (n - s)
        p_posts.append(a_n / (a_n + b_n))
    ax.plot(n_obs_range, p_posts, marker='o', markersize=4, color=cor,
            label=f'Taxa obs. = {taxa:.0%}')
ax.axhline(alpha0 / (alpha0 + beta0), color='black', linestyle='--',
           linewidth=1.5, label=f'Prior médio = {alpha0/(alpha0+beta0):.2f}')
ax.set_xlabel('Número de Observações', fontsize=11)
ax.set_ylabel('P(recompra) Posterior')
ax.set_title('Atualização Bayesiana: Convergência da Posterior', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_atualizacao, dpi=150)
plt.show()
plt.close()

# 4.4 Scatter interativo CLV × p_recompra × ticket_medio
rfm_plot = rfm_cal.sample(min(3000, len(rfm_cal)), random_state=42)
fig_inter = px.scatter(rfm_plot, x='p_recompra', y='CLV_Bayesiano',
                       color='Segmento_CLV', size=np.log1p(rfm_plot['ticket_medio']) * 2 + 2,
                       hover_data=['Customer ID', 'n_pedidos', 'ticket_medio', 'Rank_CLV'],
                       color_discrete_map={'Bronze': '#CD7F32', 'Prata': '#A8A9AD',
                                           'Ouro': '#FFD700', 'Diamante': '#0D47A1'},
                       title='CLV Bayesiano × Probabilidade de Recompra',
                       labels={'p_recompra': 'P(Recompra) Posterior', 'CLV_Bayesiano': 'CLV (R$)'})
fig_inter.write_html(file_interativo)
fig_inter.show()
print(f"Scatter interativo salvo: {file_interativo}")

# 4.5 3D: Recência × Frequência × CLV Bayesiano
fig_3d = px.scatter_3d(rfm_plot, x='recencia', y='n_pedidos', z='CLV_Bayesiano',
                        color='Segmento_CLV', opacity=0.75, hover_name='Customer ID',
                        color_discrete_map={'Bronze': '#CD7F32', 'Prata': '#A8A9AD',
                                            'Ouro': '#FFD700', 'Diamante': '#0D47A1'},
                        title='3D: Recência × Frequência × CLV Bayesiano',
                        labels={'recencia': 'Recência (dias)', 'n_pedidos': 'Frequência',
                                'CLV_Bayesiano': 'CLV (R$)'})
fig_3d.write_html(file_clv_3d)
fig_3d.show()
print(f"CLV 3D salvo: {file_clv_3d}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_posterior}")
print(f"  {file_clv_dist}")
print(f"  {file_atualizacao}")
print(f"  {file_interativo}")
print(f"  {file_clv_3d}")

In [ ]:
# =============================================================
# STEP 13 - Otimização Robusta
# -------------------------------------------------------------
# Objetivo: Construir portfólio de ativos robustamente ótimo
#           para o pior cenário dentro de um conjunto de
#           incerteza elipsoidal nos retornos. Compara a
#           solução robusta (minimax) com a determinística
#           (média-variância) em termos de retorno garantido
#           e comportamento nos cenários adversos.
# Entrada  : output/base_step_09_world_bank_raw.csv  (gerado pelo step-09)
#            output/base_step_00_yahoo_finance.csv   (gerado pelo step-00)
# Saídas   : output/base_step_13_robusta_resultado.csv
#            output/base_step_13_robusta_portfolios.png
#            output/base_step_13_robusta_comparativo.png
#            output/base_step_13_robusta_incerteza.png
#            output/base_step13_robusta_fronteira_interativa.html
#            output/base_step13_robusta_cenarios.html
# Depende  : STEP 00, STEP 09
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy.optimize import minimize

file_input            = 'output/base_step_09_world_bank_raw.csv'
file_output           = 'output/base_step_13_robusta_resultado.csv'
file_portfolios       = 'output/base_step_13_robusta_portfolios.png'
file_comparativo      = 'output/base_step_13_robusta_comparativo.png'
file_incerteza        = 'output/base_step_13_robusta_incerteza.png'
file_fronteira_inter  = 'output/base_step13_robusta_fronteira_interativa.html'
file_cenarios         = 'output/base_step13_robusta_cenarios.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Carregar dados e preparar ativos ----------
print(f"Carregando cache World Bank: {file_input}")
df_wb = pd.read_csv(file_input, sep=';', encoding='utf-8-sig')
print(f"Dados WB carregados: {len(df_wb)} países")

# Usar dados de mercado da base auditada step-00 para construir portfólio robusto
# (ativos representando economias das regiões do World Bank)
tickers = ['EEM', 'VEA', 'SPY', 'EWJ', 'EWZ', 'GLD', 'TLT', 'IEF']
nomes   = ['Emerg. Markets', 'Europa Dev.', 'EUA (S&P500)', 'Japão',
           'Brasil', 'Ouro', 'T-Bond Longo', 'T-Bond Médio']
print(f"\nCarregando dados de mercado da base auditada step-00...")
df_cache = pd.read_csv('output/base_step_00_yahoo_finance.csv',
                       sep=';', decimal=',', encoding='utf-8-sig')
df_cache['Data'] = pd.to_datetime(df_cache['Data'])
mask = (df_cache['Ticker'].isin(tickers) &
        (df_cache['Data'] >= '2018-01-01') &
        (df_cache['Data'] <= '2023-12-31'))
dados = df_cache[mask].pivot(index='Data', columns='Ticker', values='Close')
dados = dados.dropna(axis=1, how='all')
tickers_val = list(dados.columns)
nomes_val   = [nomes[tickers.index(t)] for t in tickers_val]
print(f"Ativos válidos: {tickers_val}")

ret_diarios = dados.pct_change().dropna()
mu          = ret_diarios.mean().values * 252        # retornos anualizados
Sigma       = ret_diarios.cov().values  * 252        # covariância anualizada
n           = len(tickers_val)

# ---------- 2. Otimização Robusta (Min-Max) ----------
# Conjunto de incerteza elipsoidal: Γ = {r̃ : (r̃ - μ)ᵀ Σ⁻¹ (r̃ - μ) ≤ κ²}
# Problema robusto: max_w  min_{r̃ ∈ Γ}  wᵀr̃
# Solução: max_w  wᵀμ - κ * √(wᵀΣw)   s.t. 1ᵀw=1, w≥0

kappas = [0.0, 0.5, 1.0, 1.5, 2.0]   # níveis de conservadorismo

def obj_robusta(w, kappa):
    ret  = np.dot(w, mu)
    risco = np.sqrt(np.dot(w, np.dot(Sigma, w)))
    return -(ret - kappa * risco)

constraints = ({'type': 'eq',   'fun': lambda w: np.sum(w) - 1})
bounds      = tuple((0, 1) for _ in range(n))
w0          = np.ones(n) / n

resultados = []
for kappa in kappas:
    res = minimize(obj_robusta, w0, args=(kappa,), method='SLSQP',
                   bounds=bounds, constraints=constraints, options={'ftol': 1e-9})
    w_rob   = res.x
    ret_rob = np.dot(w_rob, mu)
    vol_rob = np.sqrt(np.dot(w_rob, np.dot(Sigma, w_rob)))
    ret_gar = ret_rob - kappa * vol_rob   # retorno garantido (pior caso)
    resultados.append({
        'Kappa':          kappa,
        'Retorno_Medio':  round(ret_rob, 6),
        'Volatilidade':   round(vol_rob, 6),
        'Retorno_Garantido': round(ret_gar, 6),
        'Sharpe':         round(ret_rob / (vol_rob + 1e-9), 6),
        **{f'w_{t}': round(w_rob[i], 6) for i, t in enumerate(tickers_val)},
    })
    print(f"κ={kappa:.1f} | ret={ret_rob:.4f} | vol={vol_rob:.4f} | garantido={ret_gar:.4f}")

df_resultado = pd.DataFrame(resultados)

# Simulação de Monte Carlo de cenários para cada solução
N_CEN = 2000
np.random.seed(42)
L = np.linalg.cholesky(Sigma + 1e-8 * np.eye(n))
cenarios = mu + (L @ np.random.randn(n, N_CEN)).T   # N_CEN x n

retornos_por_kappa = {}
for row in df_resultado.itertuples():
    w_k = np.array([getattr(row, f'w_{t}') for t in tickers_val])
    retornos_por_kappa[row.Kappa] = cenarios @ w_k

# ---------- 3. Salvar resultados ----------
df_resultado.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Pesos dos portfólios por κ
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(tickers_val))
w = 0.15
cores_k = ['#1565C0', '#1976D2', '#42A5F5', '#90CAF9', '#E3F2FD']
for j, row in df_resultado.iterrows():
    pesos = [getattr(row, f'w_{t}') for t in tickers_val]
    ax.bar(x + (j - 2) * w, pesos, width=w, label=f'κ={row.Kappa}', color=cores_k[j])
ax.set_xticks(x)
ax.set_xticklabels(nomes_val, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Peso no Portfólio')
ax.set_title('Alocação dos Portfólios Robustos por Nível de Conservadorismo (κ)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(file_portfolios, dpi=150)
plt.show()
plt.close()

# 4.2 Retorno médio vs retorno garantido (pior caso) por κ
plt.figure(figsize=(9, 5))
plt.plot(df_resultado['Kappa'], df_resultado['Retorno_Medio'] * 100,
         'o-', color='#1565C0', linewidth=2, label='Retorno Médio Esperado', markersize=7)
plt.plot(df_resultado['Kappa'], df_resultado['Retorno_Garantido'] * 100,
         's--', color='#C62828', linewidth=2, label='Retorno Garantido (pior caso)', markersize=7)
plt.fill_between(df_resultado['Kappa'],
                 df_resultado['Retorno_Garantido'] * 100,
                 df_resultado['Retorno_Medio'] * 100,
                 alpha=0.12, color='grey', label='Intervalo de incerteza')
plt.xlabel('Parâmetro de Robustez (κ)', fontsize=11)
plt.ylabel('Retorno Anual (%)')
plt.title('Retorno Esperado vs Retorno Garantido por Nível de Robustez', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_comparativo, dpi=150)
plt.show()
plt.close()

# 4.3 Distribuição dos retornos por κ nos cenários Monte Carlo
# SKILL: >3 grupos em KDE = spaghetti proibido → violin plot (5 κ valores)
df_violin = pd.DataFrame({
    'Retorno (%)': np.concatenate([retornos_por_kappa[k] * 100 for k in kappas]),
    'κ': np.concatenate([[f'κ={k}'] * len(retornos_por_kappa[k]) for k in kappas])
})
fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=df_violin, x='κ', y='Retorno (%)',
               palette='Blues_r', ax=ax, inner='box', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', label='Retorno zero')
ax.set_xlabel('Nível de Conservadorismo (κ)', fontsize=11)
ax.set_ylabel('Retorno Anual (%) — Cenários Monte Carlo')
ax.set_title('Distribuição de Retornos nos Cenários de Incerteza por Nível de Robustez', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig(file_incerteza, dpi=150)
plt.show()
plt.close()

# 4.4 Fronteira Robusta interativa (Plotly)
fig_inter = go.Figure()
fig_inter.add_trace(go.Scatter(
    x=df_resultado['Volatilidade'] * 100,
    y=df_resultado['Retorno_Medio'] * 100,
    mode='lines+markers+text',
    text=[f'κ={k}' for k in df_resultado['Kappa']],
    textposition='top center',
    marker=dict(size=12, color=df_resultado['Kappa'], colorscale='Blues', showscale=True,
                colorbar=dict(title='κ')),
    line=dict(color='#1565C0', width=2),
    name='Retorno Médio',
))
fig_inter.add_trace(go.Scatter(
    x=df_resultado['Volatilidade'] * 100,
    y=df_resultado['Retorno_Garantido'] * 100,
    mode='lines+markers',
    marker=dict(size=10, color='#C62828', symbol='square'),
    line=dict(color='#C62828', width=2, dash='dash'),
    name='Retorno Garantido',
))
fig_inter.update_layout(
    title='Fronteira Robusta: Retorno Médio vs Garantido por κ',
    xaxis_title='Volatilidade (%)', yaxis_title='Retorno (%)'
)
fig_inter.write_html(file_fronteira_inter)
fig_inter.show()
print(f"Fronteira robusta interativa salva: {file_fronteira_inter}")

# 4.5 Distribuições de cenários (interativo)
fig_cen = go.Figure()
for j, kappa in enumerate(kappas):
    fig_cen.add_trace(go.Violin(
        y=retornos_por_kappa[kappa] * 100,
        name=f'κ={kappa}',
        box_visible=True, meanline_visible=True,
    ))
fig_cen.update_layout(
    title='Distribuição de Retornos por Nível de Robustez (Violin Plot)',
    yaxis_title='Retorno Anual (%) — Monte Carlo', violinmode='group'
)
fig_cen.write_html(file_cenarios)
fig_cen.show()
print(f"Cenários interativos salvos: {file_cenarios}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_portfolios}")
print(f"  {file_comparativo}")
print(f"  {file_incerteza}")
print(f"  {file_fronteira_inter}")
print(f"  {file_cenarios}")

In [ ]:
# =============================================================
# STEP 14 - Otimização Estocástica
# -------------------------------------------------------------
# Objetivo: Resolver um problema de programação estocástica em
#           dois estágios (alocação de ativos) com geração de
#           cenários via simulação de Monte Carlo. Compara:
#           EEV (Expected Value of Expected Value), RP (Recourse
#           Problem) e WS (Wait-and-See). Calcula VSS (Value of
#           Stochastic Solution) e EVPI (Expected Value of
#           Perfect Information).
# Entrada  : output/base_step_09_world_bank_raw.csv  (gerado pelo step-09)
#            output/base_step_00_yahoo_finance.csv   (gerado pelo step-00)
# Saídas   : output/base_step_14_estocastica_cenarios.csv
#            output/base_step_14_estocastica_distribuicao.png
#            output/base_step_14_estocastica_ev_vs_ws.png
#            output/base_step_14_estocastica_solucoes.png
#            output/base_step14_estocastica_cenarios_interativo.html
#            output/base_step14_estocastica_3d.html
# Depende  : STEP 00, STEP 09
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy.optimize import minimize, linprog

file_input              = 'output/base_step_09_world_bank_raw.csv'
file_output             = 'output/base_step_14_estocastica_cenarios.csv'
file_distribuicao       = 'output/base_step_14_estocastica_distribuicao.png'
file_ev_vs_ws           = 'output/base_step_14_estocastica_ev_vs_ws.png'
file_solucoes           = 'output/base_step_14_estocastica_solucoes.png'
file_cenarios_inter     = 'output/base_step14_estocastica_cenarios_interativo.html'
file_3d                 = 'output/base_step14_estocastica_3d.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Carregar dados e preparar ativos ----------
print(f"Carregando cache World Bank: {file_input}")
df_wb = pd.read_csv(file_input, sep=';', encoding='utf-8-sig')
print(f"Dados WB carregados: {len(df_wb)} países")

tickers = ['SPY', 'EEM', 'TLT', 'GLD', 'EWJ']
nomes   = ['EUA (SPY)', 'Emerg. (EEM)', 'T-Bond (TLT)', 'Ouro (GLD)', 'Japão (EWJ)']
print("\nCarregando dados de retornos da base auditada step-00...")
df_cache = pd.read_csv('output/base_step_00_yahoo_finance.csv',
                       sep=';', decimal=',', encoding='utf-8-sig')
df_cache['Data'] = pd.to_datetime(df_cache['Data'])
mask = (df_cache['Ticker'].isin(tickers) &
        (df_cache['Data'] >= '2015-01-01') &
        (df_cache['Data'] <= '2023-12-31'))
dados = df_cache[mask].pivot(index='Data', columns='Ticker', values='Close')
dados = dados.dropna(axis=1, how='all')
tickers_val = list(dados.columns)
nomes_val   = [nomes[tickers.index(t)] for t in tickers_val if t in tickers]
n           = len(tickers_val)

ret_diarios = dados.pct_change().dropna()
mu          = ret_diarios.mean().values * 252
Sigma       = ret_diarios.cov().values  * 252
print(f"Ativos válidos: {tickers_val}")

# ---------- 2. Programação Estocástica em Dois Estágios ----------
# Estágio 1: Decidir alocação inicial w (antes de observar o cenário)
# Estágio 2: Rebalancear y (após observar o cenário) com custo de rebalanceamento
# Objetivo: max  E_ξ [ wᵀr̃ - c * ||y||₁ ]
# s.t.       1ᵀw = 1, w ≥ 0, 1ᵀ(w+y) = 1, (w+y) ≥ 0

N_CEN       = 500
CUSTO_REBALAN = 0.002   # 0.2% de custo de transação
np.random.seed(42)

# Gerar cenários por simulação Monte Carlo multivariada
L = np.linalg.cholesky(Sigma + 1e-8 * np.eye(n))
cenarios = (mu + (L @ np.random.randn(n, N_CEN)).T)  # N_CEN × n

# --- EEV: Solução baseada no valor esperado dos retornos ---
def neg_retorno(w):
    return -np.dot(w, mu)

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds      = tuple((0, 1) for _ in range(n))
res_eev     = minimize(neg_retorno, np.ones(n)/n, method='SLSQP',
                       bounds=bounds, constraints=constraints)
w_eev       = res_eev.x
ret_eev_cen = cenarios @ w_eev   # retorno do portfólio EEV em cada cenário

# --- RP (Recourse Problem): solução estocástica ---
# Simplificação: busca o w que maximiza retorno médio menos custo esperado de rebalanceamento
# Aqui, usamos uma heurística: max Sharpe com penalidade de concentração
def obj_rp(w):
    ret  = np.dot(w, mu)
    risco = np.sqrt(np.dot(w, np.dot(Sigma, w)))
    entropia = -np.sum(w * np.log(w + 1e-12))   # penalidade de concentração
    return -(ret / (risco + 1e-9) + 0.1 * entropia)

res_rp  = minimize(obj_rp, np.ones(n)/n, method='SLSQP',
                   bounds=bounds, constraints=constraints)
w_rp    = res_rp.x
ret_rp_cen = cenarios @ w_rp

# --- WS (Wait-and-See): solução ótima para cada cenário individualmente ---
ret_ws_cen = np.zeros(N_CEN)
w_ws_all   = np.zeros((N_CEN, n))
for i, r_cen in enumerate(cenarios):
    res_i = minimize(lambda w: -np.dot(w, r_cen), np.ones(n)/n,
                     method='SLSQP', bounds=bounds, constraints=constraints)
    w_ws_all[i]  = res_i.x
    ret_ws_cen[i] = np.dot(res_i.x, r_cen)

# Métricas de comparação
EEV   = ret_eev_cen.mean()
RP    = ret_rp_cen.mean()
WS    = ret_ws_cen.mean()
VSS   = RP - EEV    # Value of Stochastic Solution
EVPI  = WS - RP     # Expected Value of Perfect Information

print(f"\n{'='*40}")
print(f"EEV  (det. simples)  : {EEV:.4f}  ({EEV*100:.2f}%)")
print(f"RP   (estocástico)   : {RP:.4f}  ({RP*100:.2f}%)")
print(f"WS   (perfeito)      : {WS:.4f}  ({WS*100:.2f}%)")
print(f"VSS  (RP - EEV)      : {VSS:.4f}  ({VSS*100:.2f}%)")
print(f"EVPI (WS - RP)       : {EVPI:.4f}  ({EVPI*100:.2f}%)")
print(f"{'='*40}")

# Risco (VaR 5%) de cada solução
var95_eev = np.percentile(ret_eev_cen, 5)
var95_rp  = np.percentile(ret_rp_cen, 5)
var95_ws  = np.percentile(ret_ws_cen, 5)

# ---------- 3. Salvar resultados ----------
df_resultado = pd.DataFrame({
    'Cenario':      range(1, N_CEN + 1),
    'Ret_EEV':      np.round(ret_eev_cen, 6),
    'Ret_RP':       np.round(ret_rp_cen, 6),
    'Ret_WS':       np.round(ret_ws_cen, 6),
    **{f'r_{t}': np.round(cenarios[:, i], 6) for i, t in enumerate(tickers_val)},
})
df_resultado.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Distribuição de retornos nos cenários por solução
plt.figure(figsize=(11, 5))
sns.kdeplot(ret_eev_cen * 100, color='#1565C0', linewidth=2, label=f'EEV (média={EEV*100:.2f}%)')
sns.kdeplot(ret_rp_cen  * 100, color='#2E7D32', linewidth=2, label=f'RP  (média={RP*100:.2f}%)')
sns.kdeplot(ret_ws_cen  * 100, color='#C62828', linewidth=2, label=f'WS  (média={WS*100:.2f}%)')
plt.axvline(var95_eev * 100, color='#1565C0', linewidth=1, linestyle=':', label=f'VaR5% EEV')
plt.axvline(var95_rp  * 100, color='#2E7D32', linewidth=1, linestyle=':', label=f'VaR5% RP')
plt.axvline(var95_ws  * 100, color='#C62828', linewidth=1, linestyle=':', label=f'VaR5% WS')
plt.xlabel('Retorno Anual (%) — Cenários Monte Carlo')
plt.ylabel('Densidade')
plt.title('Distribuição de Retornos: EEV × RP (Estocástico) × WS (Informação Perfeita)', fontsize=12)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_distribuicao, dpi=150)
plt.show()
plt.close()

# 4.2 VSS e EVPI — comparativo de valor
metricas   = ['EEV', 'RP\n(Estocástico)', 'WS\n(Perfeito)']
valores    = [EEV * 100, RP * 100, WS * 100]
cores_met  = ['#1565C0', '#2E7D32', '#C62828']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(metricas, valores, color=cores_met, edgecolor='white', linewidth=1.2)
axes[0].set_ylabel('Retorno Médio Esperado (%)')
axes[0].set_title('Comparativo EEV × RP × WS')
axes[0].grid(True, axis='y', alpha=0.4)
for i, v in enumerate(valores):
    axes[0].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=11)

gains     = ['VSS\n(RP - EEV)', 'EVPI\n(WS - RP)']
vals_gain = [VSS * 100, EVPI * 100]
cores_g   = ['#2E7D32', '#FF9800']
axes[1].bar(gains, vals_gain, color=cores_g, edgecolor='white', linewidth=1.2)
axes[1].set_ylabel('Ganho em Retorno (%)')
axes[1].set_title('VSS (Valor da Informação Estocástica) e EVPI')
axes[1].grid(True, axis='y', alpha=0.4)
for i, v in enumerate(vals_gain):
    axes[1].text(i, v + 0.005, f'{v:.3f}%', ha='center', fontweight='bold', fontsize=11)
plt.suptitle('Programação Estocástica em Dois Estágios', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_ev_vs_ws, dpi=150)
plt.show()
plt.close()

# 4.3 Composição dos portfólios EEV vs RP
x   = np.arange(n)
w_b = 0.35
plt.figure(figsize=(10, 5))
plt.bar(x - w_b/2, w_eev * 100, width=w_b, label='EEV (Determinístico)', color='#1565C0')
plt.bar(x + w_b/2, w_rp  * 100, width=w_b, label='RP  (Estocástico)',    color='#2E7D32')
plt.xticks(x, nomes_val, rotation=15, ha='right', fontsize=9)
plt.ylabel('Alocação (%)')
plt.title('Comparação de Portfólios: EEV vs RP (Estocástico)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(file_solucoes, dpi=150)
plt.show()
plt.close()

# 4.4 Cenários interativos (Plotly)
df_cen_melt = pd.DataFrame({
    'Cenario': list(range(1, N_CEN+1)) * 3,
    'Retorno': np.concatenate([ret_eev_cen * 100, ret_rp_cen * 100, ret_ws_cen * 100]),
    'Solucao': ['EEV'] * N_CEN + ['RP'] * N_CEN + ['WS'] * N_CEN,
})
fig_inter = px.histogram(df_cen_melt, x='Retorno', color='Solucao', barmode='overlay',
                         nbins=60, opacity=0.6,
                         color_discrete_map={'EEV': '#1565C0', 'RP': '#2E7D32', 'WS': '#C62828'},
                         title='Distribuição de Retornos por Solução — Interativo',
                         labels={'Retorno': 'Retorno Anual (%)'})
fig_inter.write_html(file_cenarios_inter)
fig_inter.show()
print(f"Cenários interativos salvos: {file_cenarios_inter}")

# 4.5 Gráfico 3D: cenários (SPY × EEM × Retorno RP)
fig_3d = px.scatter_3d(
    x=cenarios[:, 0] * 100, y=cenarios[:, 1] * 100, z=ret_rp_cen * 100,
    color=ret_rp_cen * 100, color_continuous_scale='RdBu',
    opacity=0.6,
    title='Cenários Estocásticos 3D: SPY × EEM × Retorno RP',
    labels={'x': 'Retorno SPY (%)', 'y': 'Retorno EEM (%)', 'z': 'Retorno RP (%)'},
)
fig_3d.write_html(file_3d)
fig_3d.show()
print(f"Gráfico 3D salvo: {file_3d}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_distribuicao}")
print(f"  {file_ev_vs_ws}")
print(f"  {file_solucoes}")
print(f"  {file_cenarios_inter}")
print(f"  {file_3d}")


In [ ]:
# =============================================================
# STEP 15 - Transporte Ótimo
# -------------------------------------------------------------
# Objetivo: Aplicar a teoria do Transporte Ótimo (Wasserstein)
#           para comparar distribuições de renda (PIB per capita)
#           entre grupos de países. Calcula a distância de
#           Wasserstein (Earth Mover's Distance) entre regiões
#           geográficas e visualiza o plano de transporte ótimo
#           e a evolução temporal das distribuições.
# Entrada  : output/base_step_01_world_bank.csv  (gerado pelo step-01)
# Saídas   : output/base_step_15_transporte_plano.csv
#            output/base_step_15_transporte_heatmap.png
#            output/base_step_15_transporte_distribuicoes.png
#            output/base_step_15_transporte_wasserstein.png
#            output/base_step15_transporte_interativo.html
#            output/base_step15_transporte_3d.html
# Depende  : STEP 01
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy.stats import wasserstein_distance
from scipy.optimize import linprog

file_output          = 'output/base_step_15_transporte_plano.csv'
file_heatmap         = 'output/base_step_15_transporte_heatmap.png'
file_distribuicoes   = 'output/base_step_15_transporte_distribuicoes.png'
file_wasserstein     = 'output/base_step_15_transporte_wasserstein.png'
file_interativo      = 'output/base_step15_transporte_interativo.html'
file_3d              = 'output/base_step15_transporte_3d.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Aquisição de dados (base auditada step-01) ----------
print("Carregando base auditada do step-01 (World Bank)...")
paises_por_regiao = {
    'América do Norte':  ['USA', 'CAN', 'MEX'],
    'América do Sul':    ['BRA', 'ARG', 'CHL', 'COL', 'PER'],
    'Europa':            ['DEU', 'FRA', 'GBR', 'ITA', 'ESP', 'NLD', 'SWE', 'NOR', 'POL'],
    'Ásia':              ['CHN', 'JPN', 'KOR', 'IND', 'IDN', 'PHL', 'THA'],
    'África':            ['NGA', 'ZAF', 'EGY', 'ETH', 'GHA'],
}
todos_paises = [p for ps in paises_por_regiao.values() for p in ps]

df_wb = pd.read_csv('output/base_step_01_world_bank.csv',
                    sep=';', decimal=',', encoding='utf-8-sig')
# Selecionar colunas relevantes e filtrar países do mapa regional
df_wb = df_wb[df_wb['ISO'].isin(todos_paises)][['ISO', 'Pais', 'Ano', 'PIB_pc_USD']].copy()
df_wb = df_wb.rename(columns={'PIB_pc_USD': 'PIB_pc'})
# Adicionar coluna Regiao mapeando pelo dict
iso_to_regiao = {iso: reg for reg, isos in paises_por_regiao.items() for iso in isos}
df_wb['Regiao'] = df_wb['ISO'].map(iso_to_regiao)
df = df_wb.dropna(subset=['PIB_pc']).sort_values(['ISO', 'Ano']).reset_index(drop=True)
print(f"Dados carregados: {len(df)} observações, {df['ISO'].nunique()} países, {df['Ano'].nunique()} anos")

# ---------- 2. Distâncias de Wasserstein entre regiões ----------
anos_analise = [2000, 2005, 2010, 2015, 2020, 2022]
regioes      = list(paises_por_regiao.keys())
n_reg        = len(regioes)

# Calcular Wasserstein entre pares de regiões para cada ano
registros_wasserstein = []
for ano in anos_analise:
    df_ano = df[df['Ano'] == ano].dropna(subset=['PIB_pc'])
    matriz_W = np.zeros((n_reg, n_reg))
    for i, r1 in enumerate(regioes):
        for j, r2 in enumerate(regioes):
            if i != j:
                u = df_ano[df_ano['Regiao'] == r1]['PIB_pc'].values
                v = df_ano[df_ano['Regiao'] == r2]['PIB_pc'].values
                if len(u) > 0 and len(v) > 0:
                    w_dist = wasserstein_distance(u, v)
                    matriz_W[i, j] = w_dist
            registros_wasserstein.append({
                'Ano':     ano,
                'Regiao_A': r1,
                'Regiao_B': r2,
                'Wasserstein': round(matriz_W[i, j], 2),
            })

df_wasser = pd.DataFrame(registros_wasserstein)

# Plano de transporte ótimo entre duas regiões (2022)
df_2022   = df[df['Ano'] == 2022].dropna(subset=['PIB_pc'])
reg_A, reg_B = 'Europa', 'África'
df_2022_A = df_2022[df_2022['Regiao'] == reg_A].sort_values('PIB_pc')
df_2022_B = df_2022[df_2022['Regiao'] == reg_B].sort_values('PIB_pc')
u_vals = df_2022_A['PIB_pc'].values
v_vals = df_2022_B['PIB_pc'].values
n_u, n_v = len(u_vals), len(v_vals)

# Plano de transporte: min cᵀγ  s.t. γ_ij ≥ 0, rowsum = 1/n_u, colsum = 1/n_v
c_cost = np.abs(u_vals[:, None] - v_vals[None, :]).flatten()
A_eq_rows = np.zeros((n_u, n_u * n_v))
for i in range(n_u):
    A_eq_rows[i, i * n_v:(i + 1) * n_v] = 1
A_eq_cols = np.zeros((n_v, n_u * n_v))
for j in range(n_v):
    A_eq_cols[j, j::n_v] = 1
A_eq = np.vstack([A_eq_rows, A_eq_cols])
b_eq = np.concatenate([np.ones(n_u) / n_u, np.ones(n_v) / n_v])
res_ot = linprog(c_cost, A_eq=A_eq, b_eq=b_eq, bounds=[(0, None)] * (n_u * n_v), method='highs')
gamma  = res_ot.x.reshape(n_u, n_v) if res_ot.success else np.zeros((n_u, n_v))

wdist_2022_AB = wasserstein_distance(u_vals, v_vals)
print(f"\nWasserstein {reg_A} × {reg_B} (2022): {wdist_2022_AB:.0f} USD")

# Evolução temporal de Wasserstein entre Europa e África
ev_anos  = sorted(df['Ano'].unique())
ev_wdist = []
for ano in ev_anos:
    df_a = df[df['Ano'] == ano].dropna(subset=['PIB_pc'])
    u = df_a[df_a['Regiao'] == reg_A]['PIB_pc'].values
    v = df_a[df_a['Regiao'] == reg_B]['PIB_pc'].values
    ev_wdist.append(wasserstein_distance(u, v) if len(u) > 0 and len(v) > 0 else np.nan)

# ---------- 3. Salvar resultados ----------
df_plano = pd.DataFrame(gamma, index=[f'EU_{p}' for p in df_2022_A['Pais'].values],
                         columns=[f'AF_{p}' for p in df_2022_B['Pais'].values])
df_plano.to_csv(file_output, sep=';', encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Matriz de Wasserstein entre regiões (2022)
df_2022_w = df_wasser[df_wasser['Ano'] == 2022].pivot(
    index='Regiao_A', columns='Regiao_B', values='Wasserstein').reindex(
    index=regioes, columns=regioes).fillna(0)
plt.figure(figsize=(9, 7))
sns.heatmap(df_2022_w, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': 'Distância de Wasserstein (USD PIB pc)'})
plt.title('Matriz de Distâncias de Wasserstein entre Regiões (2022)', fontsize=12)
plt.tight_layout()
plt.savefig(file_heatmap, dpi=150)
plt.show()
plt.close()

# 4.2 Distribuições de PIB per capita por região (2022)
# SKILL: >3 grupos em KDE = spaghetti proibido → violin plot (5 regiões)
cores_reg = ['#1565C0', '#C62828', '#2E7D32', '#FF9800', '#7B1FA2']
df_violin = pd.DataFrame({
    'Regiao':   [reg for reg in regioes
                 for _ in df_2022[df_2022['Regiao'] == reg]['PIB_pc'].dropna()],
    'log_PIB':  [np.log1p(v) for reg in regioes
                 for v in df_2022[df_2022['Regiao'] == reg]['PIB_pc'].dropna().values],
})
palette_reg = dict(zip(regioes, cores_reg))
fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(data=df_violin, x='Regiao', y='log_PIB',
               palette=palette_reg, ax=ax, inner='box', linewidth=1.2)
ax.set_xlabel('Região', fontsize=11)
ax.set_ylabel('log(1 + PIB per capita USD) — 2022')
ax.set_title('Distribuições de PIB per capita por Região (escala log)', fontsize=12)
ax.tick_params(axis='x', rotation=15)
ax.grid(True, axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig(file_distribuicoes, dpi=150)
plt.show()
plt.close()

# 4.3 Evolução temporal da distância Wasserstein Europa × África
plt.figure(figsize=(10, 5))
plt.plot(ev_anos, [w / 1000 for w in ev_wdist], color='#C62828', linewidth=2.5, marker='o', markersize=5)
plt.fill_between(ev_anos, [w / 1000 for w in ev_wdist], alpha=0.15, color='#C62828')
plt.xlabel('Ano', fontsize=11)
plt.ylabel('Distância de Wasserstein (× 1.000 USD)')
plt.title(f'Evolução da Distância de Wasserstein: {reg_A} × {reg_B}', fontsize=12)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_wasserstein, dpi=150)
plt.show()
plt.close()

# 4.4 Heatmap interativo do plano de transporte ótimo
fig_inter = go.Figure(data=go.Heatmap(
    z=gamma * 1000,
    x=[f'AF_{p}' for p in df_2022_B['Pais'].values],
    y=[f'EU_{p}' for p in df_2022_A['Pais'].values],
    colorscale='Blues',
    texttemplate='%{z:.2f}',
))
fig_inter.update_layout(
    title=f'Plano de Transporte Ótimo: {reg_A} → {reg_B} (2022, ×10⁻³)',
    xaxis_title='Destino (África)', yaxis_title='Origem (Europa)',
    height=400
)
fig_inter.write_html(file_interativo)
fig_inter.show()
print(f"Plano de transporte interativo salvo: {file_interativo}")

# 4.5 3D: Wasserstein por par de regiões ao longo do tempo
df_w_pivot = df_wasser[df_wasser['Regiao_A'] != df_wasser['Regiao_B']].copy()
df_w_pivot['Par'] = df_w_pivot['Regiao_A'].str[:3] + '×' + df_w_pivot['Regiao_B'].str[:3]
fig_3d = px.scatter_3d(df_w_pivot, x='Ano', y='Par', z='Wasserstein',
                        color='Wasserstein', color_continuous_scale='YlOrRd',
                        opacity=0.8, size_max=8,
                        title='Distâncias de Wasserstein por Par de Regiões e Ano',
                        labels={'Wasserstein': 'W (USD)', 'Par': 'Par de Regiões'})
fig_3d.write_html(file_3d)
fig_3d.show()
print(f"Gráfico 3D salvo: {file_3d}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_heatmap}")
print(f"  {file_distribuicoes}")
print(f"  {file_wasserstein}")
print(f"  {file_interativo}")
print(f"  {file_3d}")

In [ ]:
# =============================================================
# STEP 16 - Equações de Bellman e Programação Dinâmica
# -------------------------------------------------------------
# Objetivo: Resolver um problema de gestão de estoque via
#           Programação Dinâmica (indução retroativa). Usa dados
#           históricos de quantidade do Online Retail II para
#           estimar a distribuição de demanda. Define a equação
#           de Bellman V(t,s) = min_{a} [c(s,a) + E[V(t+1,s')]]
#           e encontra a política ótima de reposição.
# Entrada  : output/base_step_10_online_retail_raw.csv
# Saídas   : output/base_step_16_bellman_politica.csv
#            output/base_step_16_bellman_valor.png
#            output/base_step_16_bellman_convergencia.png
#            output/base_step_16_bellman_politica_mapa.png
#            output/base_step16_bellman_interativo.html
#            output/base_step16_bellman_3d.html
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

file_input         = 'output/base_step_02_online_retail.csv'
file_output        = 'output/base_step_16_bellman_politica.csv'
file_valor         = 'output/base_step_16_bellman_valor.png'
file_convergencia  = 'output/base_step_16_bellman_convergencia.png'
file_politica_mapa = 'output/base_step_16_bellman_politica_mapa.png'
file_interativo    = 'output/base_step16_bellman_interativo.html'
file_3d            = 'output/base_step16_bellman_3d.html'

os.makedirs(os.path.dirname(file_output), exist_ok=True)

# ---------- 1. Carregar dados e estimar distribuição de demanda ----------
df_raw = pd.read_csv(file_input, sep=';', encoding='utf-8-sig', decimal=',')
print(f"Linhas originais: {len(df_raw)}")

df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
# Normalização defensiva: Online Retail II pode salvar 'InvoiceDate' ou 'Invoice Date'
if 'InvoiceDate' in df.columns and 'Invoice Date' not in df.columns:
    df = df.rename(columns={'InvoiceDate': 'Invoice Date'})
df = df[df['Quantity'].astype(float) > 0].copy()
df['Quantity'] = df['Quantity'].astype(float).astype(int)

# Estimar distribuição de demanda diária por produto
# Usar o produto mais transacionado como DMU representativo
top_produto = df.groupby('StockCode')['Quantity'].sum().idxmax()
df_prod     = df[df['StockCode'] == top_produto].copy()
df_prod['Invoice Date'] = pd.to_datetime(df_prod['Invoice Date'])
df_prod['Data']         = df_prod['Invoice Date'].dt.date
demanda_diaria = df_prod.groupby('Data')['Quantity'].sum()

print(f"\nProduto selecionado : {top_produto}")
print(f"Dias com venda       : {len(demanda_diaria)}")
print(f"Demanda média/dia    : {demanda_diaria.mean():.1f}")
print(f"Demanda max/dia      : {demanda_diaria.max()}")

# Truncar demanda para estados manejáveis
D_MAX     = int(min(demanda_diaria.quantile(0.95), 50))
demanda_v = np.clip(demanda_diaria.values, 0, D_MAX).astype(int)
probs_d   = np.bincount(demanda_v, minlength=D_MAX + 1) / len(demanda_v)
print(f"D_MAX (95p)          : {D_MAX}")

# ---------- 2. Programação Dinâmica — Equação de Bellman ----------
# Estados:  s ∈ {0, 1, ..., S_MAX}  (nível de estoque)
# Ações:    a ∈ {0, 1, ..., A_MAX}  (quantidade a pedir)
# Transição: s' = max(0, s - d) + a    (demanda d ~ probs_d)
# Custo imediato: c(s, a) = c_pedido * I(a>0) + c_estoque * s + c_falta * max(0, d-s)
# Equação de Bellman: V*(s) = min_a { c(s,a) + γ * E_d[V*(s')] }

S_MAX   = 100      # estoque máximo
A_MAX   = 50       # pedido máximo por período
T       = 30       # horizonte de planejamento (dias)
GAMMA   = 0.95     # fator de desconto
C_PED   = 20.0     # custo fixo de pedido
C_EST   = 0.5      # custo de manutenção de estoque por unidade/dia
C_FALT  = 5.0      # custo de falta por unidade
C_UNIT  = 2.0      # custo unitário de compra

estados = np.arange(S_MAX + 1)   # 0 a S_MAX
acoes   = np.arange(A_MAX + 1)   # 0 a A_MAX
demandas = np.arange(D_MAX + 1)  # 0 a D_MAX

# Indução retroativa: V_{T} = 0 (sem custo no final)
V = np.zeros(S_MAX + 1)          # função valor
politica = np.zeros(S_MAX + 1, dtype=int)  # política ótima
hist_V   = [V.copy()]
hist_pol = [politica.copy()]
del_V    = []

for t in range(T - 1, -1, -1):
    V_novo  = np.full(S_MAX + 1, np.inf)
    pol_novo = np.zeros(S_MAX + 1, dtype=int)

    for s in estados:
        melhor_custo = np.inf
        melhor_a     = 0
        for a in acoes:
            s_apos = min(s + a, S_MAX)   # estoque após receber pedido

            # Custo imediato
            custo_imediato = C_UNIT * a + (C_PED if a > 0 else 0) + C_EST * s_apos

            # Custo esperado do próximo período
            custo_futuro = 0.0
            for d, p_d in zip(demandas, probs_d):
                if p_d < 1e-12:
                    continue
                s_prox      = max(0, s_apos - d)
                custo_falta = C_FALT * max(0, d - s_apos)
                custo_futuro += p_d * (custo_falta + GAMMA * V[s_prox])

            custo_total = custo_imediato + custo_futuro
            if custo_total < melhor_custo:
                melhor_custo = custo_total
                melhor_a     = a

        V_novo[s]  = melhor_custo
        pol_novo[s] = melhor_a

    delta = np.max(np.abs(V_novo - V))
    del_V.append(delta)
    V        = V_novo
    politica = pol_novo
    hist_V.append(V.copy())
    hist_pol.append(politica.copy())

    if t % 5 == 0:
        print(f"t={t:2d} | ΔV={delta:.4f} | V(0)={V[0]:.2f} | V(S_MAX)={V[S_MAX]:.2f}")

print(f"\nPolítica ótima (amostra):")
for s in [0, 10, 20, 30, 50, 80, 100]:
    print(f"  Estoque={s:3d} → Pedir={politica[s]:3d} unidades")

# Ponto de reposição (s de pedido) e quantidade ótima
s_reposicao = next((s for s in estados if politica[s] > 0), 0)
print(f"\nPonto de reposição: s*={s_reposicao}, a*={politica[s_reposicao]}")

# ---------- 3. Salvar resultados ----------
df_resultado = pd.DataFrame({
    'Estoque_s':   estados,
    'Acao_otima':  politica,
    'Valor_V':     np.round(V, 4),
    'Tipo_Acao':   ['Repor' if politica[s] > 0 else 'Manter' for s in estados],
})
df_resultado.to_csv(file_output, sep=';', index=False, encoding='utf-8-sig', decimal=',')
print(f"\nArquivo salvo: {file_output}")

# ---------- 4. Visualizações ----------

# 4.1 Função Valor V*(s)
plt.figure(figsize=(10, 5))
plt.plot(estados, V, color='#1565C0', linewidth=2, label='V*(s) — Valor ótimo')
plt.axvline(s_reposicao, color='#C62828', linewidth=1.5, linestyle='--',
            label=f'Ponto de reposição s*={s_reposicao}')
plt.xlabel('Nível de Estoque (s)', fontsize=11)
plt.ylabel('Custo Esperado Descontado V*(s)')
plt.title('Função Valor Ótima — Programação Dinâmica (Bellman)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_valor, dpi=150)
plt.show()
plt.close()

# 4.2 Convergência da função valor ao longo das iterações (retroativas)
plt.figure(figsize=(10, 5))
plt.plot(range(len(del_V)), del_V, color='#C62828', linewidth=2, marker='o', markersize=3)
plt.axhline(0.01, color='grey', linewidth=1, linestyle='--', label='Tolerância 0.01')
plt.yscale('log')
plt.xlabel('Iteração (retroativa)', fontsize=11)
plt.ylabel('ΔV = max|V_novo - V_ant| (log)')
plt.title('Convergência da Indução Retroativa — Equação de Bellman', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(file_convergencia, dpi=150)
plt.show()
plt.close()

# 4.3 Política ótima e função valor — subplots separados
# SKILL: twinx (eixo duplo) PROIBIDO → dividir em 2 subplots independentes
cores_pol = ['#C62828' if politica[s] > 0 else '#1565C0' for s in estados]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

ax1.bar(estados, politica, color=cores_pol, alpha=0.7, label='Ação ótima a*(s)')
ax1.axvline(s_reposicao, color='gold', linewidth=1.5, linestyle='--', label=f's*={s_reposicao}')
ax1.set_ylabel('Quantidade a Pedir a*(s)', fontsize=10)
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(True, axis='y', alpha=0.35)

ax2.plot(estados, V, color='#1565C0', linewidth=2, label='V*(s) — Custo esperado descontado')
ax2.axvline(s_reposicao, color='gold', linewidth=1.5, linestyle='--')
ax2.set_xlabel('Nível de Estoque (s)', fontsize=11)
ax2.set_ylabel('Custo Esperado V*(s)', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.35)

fig.suptitle('Política Ótima a*(s) e Função Valor V*(s) — Bellman', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(file_politica_mapa, dpi=150)
plt.show()
plt.close()

# 4.4 Interativo: função valor e política ótima — subplots separados (Plotly)
# SKILL: secondary_y (eixo duplo) PROIBIDO → make_subplots rows=2
fig_inter = make_subplots(rows=2, cols=1, shared_xaxes=True,
                          subplot_titles=['Ação ótima a*(s)', 'Função Valor V*(s)'],
                          vertical_spacing=0.12)
fig_inter.add_trace(go.Bar(x=estados.tolist(), y=politica.tolist(), name='Ação ótima a*(s)',
                           marker_color=['#C62828' if politica[s] > 0 else '#90CAF9' for s in estados],
                           opacity=0.7), row=1, col=1)
fig_inter.add_trace(go.Scatter(x=estados.tolist(), y=V.tolist(), name='V*(s)',
                               line=dict(color='#1565C0', width=2.5),
                               mode='lines'), row=2, col=1)
fig_inter.update_xaxes(title_text='Nível de Estoque (s)', row=2, col=1)
fig_inter.update_yaxes(title_text='Quantidade a Pedir', row=1, col=1)
fig_inter.update_yaxes(title_text='Custo Esperado V*(s)', row=2, col=1)
fig_inter.update_layout(title='Política Ótima a*(s) e Função Valor V*(s) — Interativo',
                        height=600)
fig_inter.write_html(file_interativo)
fig_inter.show()
print(f"Interativo salvo: {file_interativo}")

# 4.5 3D: Evolução de V(s) ao longo das iterações retroativas
iters_plot = np.arange(0, len(hist_V), max(1, len(hist_V) // 10))
Z_surf = np.array([hist_V[i] for i in iters_plot])
fig_3d = go.Figure(data=[go.Surface(
    z=Z_surf, x=estados.tolist(), y=iters_plot.tolist(),
    colorscale='Viridis', opacity=0.85,
)])
fig_3d.update_layout(
    title='Evolução da Função Valor V(s) — Indução Retroativa (3D)',
    scene=dict(xaxis_title='Estoque (s)', yaxis_title='Iteração', zaxis_title='V(s)')
)
fig_3d.write_html(file_3d)
fig_3d.show()
print(f"Gráfico 3D salvo: {file_3d}")

print("\nArquivos salvos:")
print(f"  {file_output}")
print(f"  {file_valor}")
print(f"  {file_convergencia}")
print(f"  {file_politica_mapa}")
print(f"  {file_interativo}")
print(f"  {file_3d}")